# AIML Capstone Project - Industrial Safety and Health Analytics Database

<a id="table-of-contents"></a>
## Table of Contents

1. [Overview](#overview)
2. [Import the necessary libraries](#import-libraries)
3. [Data Collection](#data-collection)
4. [Data Cleansing](#data-cleansing)
5. [Data Pre-processing](#data-preprocessing)
6. [EDA (Data Analysis and Preparation)](#eda)
    - [Univariate Analysis](#univariate-analysis)
    - [Bivariate Analysis and Hypothesis testing](#bivariate-analysis)
7. [NLP Analysis](#nlp-analysis)
8. [NLP Pre-processing](#nlp-pre-processing)
    - [Word Cloud](#wordcloud)
9. [Feature Engineering](#feature-engineering)
10. [Design, train and test machine learning classifiers](#ml-models)
    - [All models - Original data](#ml-models-original-data)
    - [All models - Oversampling data](#ml-models-oversampling-data)
    - [All models - SMOTE data](#ml-models-smote-data)
    - [Hyperparameter tuning with original features](#ml-models-hyperparam-tuning)
    - [Bootstrap Sampling](#ml-models-bootstrap-sampling)
11. [Design, train and test Neural networks classifiers](#ann-models)
    - [Convert Classification to Numeric problem](#ann-models-clas-to-num-problem)
    - [Multiclass classification - Target variable - One hot encoded](#ann-models-multi-class)
12. [Design, train and test RNN or LSTM classifiers](#nlp-models)
    - [Creating a Model with Text Inputs Only](#nlp-models-text-input)
    - [Creating a Model with Categorical features Only](#nlp-models-cat-features)
    - [Creating a Model with Multiple Inputs](#nlp-models-multiple-input)
13. [Conclusion](#conclusion)
14. [Recommendations](#recommendations)
15. [Limitations](#limitations)
16. [Closing Reflections](#closing-reflections)
17. [References](#references)

<a id="overview"></a>
## 1. Overview

### Domain - Industrial safety. NLP based Chatbot.

### Context:

The database comes from one of the biggest industry in Brazil and in the world. It is an urgent need for industries/companies around the
globe to understand why employees still suffer some injuries/accidents in plants. Sometimes they also die in such environment.

### Data Description:

This The database is basically records of accidents from 12 different plants in 03 different countries which every line in the data is an
occurrence of an accident.

**Columns description:**

- **Data**: timestamp or time/date information
- **Countries**: which country the accident occurred (anonymised)
- **Local**: the city where the manufacturing plant is located (anonymised)
- **Industry sector**: which sector the plant belongs to
- **Accident level**: from I to VI, it registers how severe was the accident (I means not severe but VI means very severe)
- **Potential Accident Level**: Depending on the Accident Level, the database also registers how severe the accident could have been (due to other factors
involved in the accident)
- **Genre**: if the person is male of female
- **Employee or Third Party**: if the injured person is an employee or a third party
- **Critical Risk**: some description of the risk involved in the accident
- **Description**: Detailed description of how the accident happened.

Link to download the dataset: https://www.kaggle.com/ihmstefanini/industrial-safety-and-health-analytics-database

[Table of Contents](#table-of-contents)

<a id="import-libraries"></a>
## 2. Import the necessary libraries

In [ ]:
import numpy as np
np.random.seed(7)

Firstly, let's select TensorFlow version 2.x in colab

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

import random, re
import time

# used to supress display of warnings
import warnings

import missingno as mno

# nlp libraries
import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from tqdm import tqdm
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
from sklearn.feature_extraction.text import TfidfVectorizer

import holoviews as hv
from holoviews import opts

import os;
from os import makedirs

# sampling methods
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

# import zscore for scaling the data
from scipy.stats import zscore

from scipy.stats import randint as sp_randint

# save models
import pickle

# pre-processing methods
from sklearn.model_selection import train_test_split

# the classification models 
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# ensemble models
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# methods and classes for evaluation
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score, precision_score, roc_auc_score

# cross-validation methods
from sklearn.model_selection import KFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# feature selection methods
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV

# pre-processing methods
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import LabelEncoder

# Deep learning libraries
from keras.utils import np_utils
from keras.utils import plot_model
from keras.layers import Input
from keras.layers.merge import Concatenate
from keras.optimizers import SGD
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from keras.models import Model
from tensorflow.keras.layers import Flatten, Activation, Dense, LSTM, BatchNormalization, Embedding, Dropout, Flatten, Bidirectional, GlobalMaxPool1D
from keras.models import model_from_json
from keras.regularizers import l1, l2, l1_l2
from keras.constraints import maxnorm, min_max_norm
from keras.constraints import unit_norm
from keras.callbacks import ReduceLROnPlateau
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from keras.models import model_from_json

from keras.models import load_model
from keras.wrappers.scikit_learn import KerasClassifier

# Keras pre-processing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
os.environ['HV_DOC_HTML'] = 'true'

def _render(self, **kw):
  hv.extension('bokeh')
  return hv.Store.render(self)
hv.core.Dimensioned._repr_mimebundle_ = _render

In [ ]:
os.environ['PYTHONHASHSEED']=str(7)

# Reproduce the results
def reset_random_seeds():
   os.environ['PYTHONHASHSEED']=str(7)
   #np.random.seed(7)
   #random.seed(7)
   tf.random.set_seed(7)

#random_state = 42
#np.random.seed(random_state)
#tf.random.set_seed(random_state)

!rm -R log/

#### Setting Options

In [ ]:
# suppress display of warnings
warnings.filterwarnings('ignore')

# display all dataframe columns
pd.options.display.max_columns = None

# to set the limit to 3 decimals
pd.options.display.float_format = '{:.7f}'.format

# display all dataframe rows
pd.options.display.max_rows = None

[Table of Contents](#table-of-contents)

<a id="data-collection"></a>
## 3. Data Collection

In [ ]:
# Read IHMStefanini_industrial_safety_and_health_database_with_accidents_description.csv file
industry_df = pd.read_csv("../input/industrial-safety-and-health-analytics-database/IHMStefanini_industrial_safety_and_health_database_with_accidents_description.csv")

In [ ]:
# Get the top 5 rows
display(industry_df.head())

#### Shape of the data

In [ ]:
print("Number of rows = {0} and Number of Columns = {1} in the Data frame".format(industry_df.shape[0], industry_df.shape[1]))

#### Data type of each attribute

In [ ]:
# Check datatypes
industry_df.dtypes

* **From the above output, we see that except first column all other columns datatype is object.**

* **Categorical columns - 'Countries', 'Local', 'Industry Sector', 'Accident Level', 'Potential Accident Level', 'Genre', 'Employee or Third Party', 'Critical Risk', 'Description'**

* **Date column - 'Data'**

In [ ]:
# Check Data frame info
industry_df.info()

In [ ]:
# Column names of Data frame
industry_df.columns

## Data Collection Summary:

1. There are about 425 rows and 11 columns in the dataset.
2. We noticed that except a 'date' column all other columns are categorical columns.

[Table of Contents](#table-of-contents)

<a id="data-cleansing"></a>
## 4. Data Cleansing

#### Remove 'Unnamed: 0' and Rename - 'Data', 'Countries', 'Genre', 'Employee or Third Party' columns

In [ ]:
# Remove 'Unnamed: 0' column from Data frame
industry_df.drop("Unnamed: 0", axis=1, inplace=True)

# Rename 'Data', 'Countries', 'Genre', 'Employee or Third Party' columns in Data frame
industry_df.rename(columns={'Data':'Date', 'Countries':'Country', 'Genre':'Gender', 'Employee or Third Party':'Employee type'}, inplace=True)

# Get the top 2 rows
industry_df.head(2)

#### Check Duplicates

In [ ]:
# Check duplicates in a data frame
industry_df.duplicated().sum()

In [ ]:
# View the duplicate records
duplicates = industry_df.duplicated()

industry_df[duplicates]

* There is no need to worry about preserving the data; it is already a part of the industry dataset and we can merely remove or drop these rows from your cleaned data

#### Drop Duplicates

In [ ]:
# Delete duplicate rows
industry_df.drop_duplicates(inplace=True)

In [ ]:
# Get the shape of Industry data
industry_df.shape

In [ ]:
print("Number of rows = {0} and Number of Columns = {1} in the Data frame after removing the duplicates.".format(industry_df.shape[0], industry_df.shape[1]))

#### Check Outliers

As we know, there is no concept of outliers detection in categorical variables(nominal and ordinal), as each value is count as labels. Let's check the unique and frequency(mode) of each variable.

In [ ]:
# Check unique values of all columns except 'Description' column
for x in industry_df.columns:
    if x != 'Description':
      print('--'*30); print(f'Unique values of "{x}" column'); print('--'*30)
      print(industry_df[x].unique())
      print('\n')

* We observed that there are records of accidents from 1st Jan 2016 to 9th July 2017 in every month. So there are no outliers in the 'Date' column.

* There are only three country types so there are no outliers in 'Country' column.

* There are 12 Local cities where manufacturing plant is located and it's types are in sequence so there are no outliers in 'Local' column.

* There are only three Industry Sector types which are in sequence so there are no outliers in 'Industry Sector' column.

* There are only five Accident Level types which are in sequence so there are no outliers in 'Accident Level' column.

* There are only six Potential Accident Level types which are in sequence so there are no outliers in 'Potential Accident Level' column.

* There are only two Gender types in the provided data so there are no outliers in 'Gender' column.

* There are only three Employee types in the provided data so there are no outliers in 'Gender' column.

* There are quite a lot of Critical risk descriptions and we don't see any outliers but with the help of SME we can decide whether this column has outliers or not.

#### Check Missing Values

In [ ]:
# Check the presence of missing values
industry_df.isnull().sum()

In [ ]:
# Visualize missing values
mno.matrix(industry_df, figsize = (20, 6));

## Data Cleansing Summary:

1. Removed 'Unnamed: 0' column and renamed - 'Data', 'Countries', 'Genre', 'Employee or Third Party' columns in the dataset.
2. We had 7 duplicate instances in the dataset and dropped those duplicates.
3. There are no outliers in the dataset.
4. No missing values in dataset.
5. We are left with 418 rows and 10 columns after data cleansing.

[Table of Contents](#table-of-contents)

<a id="data-pre-processing"></a>
## 5. Data Pre-processing

* To better understand the data, I am extracting the day, month and year from Date column and creating new features such as weekday, weekofyear.

In [ ]:
industry_df['Date'] = pd.to_datetime(industry_df['Date'])

industry_df['Year'] = industry_df.Date.apply(lambda x : x.year)
industry_df['Month'] = industry_df.Date.apply(lambda x : x.month)
industry_df['Day'] = industry_df.Date.apply(lambda x : x.day)
industry_df['Weekday'] = industry_df.Date.apply(lambda x : x.day_name())
industry_df['WeekofYear'] = industry_df.Date.apply(lambda x : x.weekofyear)

industry_df.head()

As we know, this database comes from one of the biggest industry in Brazil which has four climatological seasos as below.

https://seasonsyear.com/Brazil

* Spring : September to November
* Summer : December to February
* Autumn : March to May
* Winter : June to August

We can create seasonal variable based on month variable.

In [ ]:
# function to create month variable into seasons
def month2seasons(x):
    if x in [9, 10, 11]:
        season = 'Spring'
    elif x in [12, 1, 2]:
        season = 'Summer'
    elif x in [3, 4, 5]:
        season = 'Autumn'
    elif x in [6, 7, 8]:
        season = 'Winter'
    return season

In [ ]:
industry_df['Season'] = industry_df['Month'].apply(month2seasons)
industry_df.head(3)

We can create holidays variable based on Brazil holidays list from 2016 and 2017.

Another national holidays are election days. There are a plenty of unofficial ethnic and religious holidays in Brazil. Octoberfest, Brazilian Carnival, Kinderfest, Fenaostra, Fenachopp, Musikfest, Schutzenfest, Kegelfest, Cavalhadas, Oberlandfest, Tirolerfest, Marejada are among them.

**Note**: Considering official holidays only.

In [ ]:
import holidays

brazil_holidays = []

print('--'*40); print('List of Brazil holidays in 2016'); print('--'*40)
for date in holidays.Brazil(years = 2016).items():
    brazil_holidays.append(str(date[0]))
    print(date)

print('--'*40); print('List of Brazil holidays in 2017'); print('--'*40)
for date in holidays.Brazil(years = 2017).items():
    brazil_holidays.append(str(date[0]))
    print(date)

In [ ]:
industry_df['Is_Holiday'] = [1 if str(val).split()[0] in brazil_holidays else 0 for val in industry_df['Date']]
industry_df.head(3)

[Table of Contents](#table-of-contents)

<a id="eda"></a>
## 6. EDA (Data Analysis and Preparation)

#### Variable Identification

* **Target variable:** 'Accident Level', 'Potential Accident Level'
* **Predictors (Input varibles):** 'Date', 'Country', 'Local', 'Industry Sector', 'Gender', 'Employee type', 'Critical Risk', 'Description'

<a id="univariate-analysis"></a>
#### Univariate Analysis

**Country**

In [ ]:
print('--'*30); print('Value Counts for `Country` label'); print('--'*30)

total_row_cnt = industry_df.shape[0]
country_01_cnt = industry_df[industry_df.Country == 'Country_01'].shape[0]
country_02_cnt = industry_df[industry_df.Country == 'Country_02'].shape[0]
country_03_cnt = industry_df[industry_df.Country == 'Country_03'].shape[0]

print(f'Country_01 count: {country_01_cnt} i.e. {round(country_01_cnt/total_row_cnt*100, 0)}%')
print(f'Country_02 count: {country_02_cnt} i.e. {round(country_02_cnt/total_row_cnt*100, 0)}%')
print(f'Country_03 count: {country_03_cnt} i.e. {round(country_03_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Distributon of `Country` label'); print('--'*30)
_ = industry_df['Country'].value_counts().plot(kind = 'pie', autopct = '%.0f%%', labels = ['Country_01', 'Country_02', 'Country_03'], figsize = (10, 6))

* 59% accidents occurred in Country_01
* 31% accidents occurred in Country_02
* 10% accidents occurred in Country_03



**Local**

In [ ]:
local_cnt = np.round(industry_df['Local'].value_counts(normalize=True) * 100)

hv.extension('bokeh')
hv.Bars(local_cnt).opts(title="Local Count", color="#8888ff", xlabel="Locals", ylabel="Percentage", yformatter='%d%%')\
                .opts(opts.Bars(width=700, height=300,tools=['hover'],show_grid=True))

* Highest manufacturing plants are located in Local_03 city.
* Lowest manufacturing plants are located in Local_09 city.

**Industry Sector**

In [ ]:
print('--'*30); print('Value Counts for `Industry Sector` label'); print('--'*30)

Mining_cnt = industry_df[industry_df['Industry Sector'] == 'Mining'].shape[0]
Metals_cnt = industry_df[industry_df['Industry Sector'] == 'Metals'].shape[0]
Others_cnt = industry_df[industry_df['Industry Sector'] == 'Others'].shape[0]

print(f'Mining count: {Mining_cnt} i.e. {round(Mining_cnt/total_row_cnt*100, 0)}%')
print(f'Metals count: {Metals_cnt} i.e. {round(Metals_cnt/total_row_cnt*100, 0)}%')
print(f'Others count: {Others_cnt} i.e. {round(Others_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Distributon of `Industry Sector` label'); print('--'*30)

sector_cnt = np.round(industry_df['Industry Sector'].value_counts(normalize=True) * 100)

hv.Bars(sector_cnt).opts(title="Industry Sector Count", color="#8888ff", xlabel="Sectors", ylabel="Percentage", yformatter='%d%%')\
                .opts(opts.Bars(width=500, height=300,tools=['hover'],show_grid=True))\
                * hv.Text('Mining', 15, f"{int(sector_cnt.loc['Mining'])}%")\
                * hv.Text('Metals', 15, f"{int(sector_cnt.loc['Metals'])}%")\
                * hv.Text('Others', 15, f"{int(sector_cnt.loc['Others'])}%")

* 57% manufacturing plants belongs to Mining sector.
* 32% manufacturing plants belongs to Metals sector.
* 11% manufacturing plants belongs to Others sector.

**Accident Levels**

In [ ]:
print('--'*30); print('Value Counts for `Accident Level` label'); print('--'*40)

I_acc_cnt = industry_df[industry_df['Accident Level'] == 'I'].shape[0]
II_acc_cnt = industry_df[industry_df['Accident Level'] == 'II'].shape[0]
III_acc_cnt = industry_df[industry_df['Accident Level'] == 'III'].shape[0]
IV_acc_cnt = industry_df[industry_df['Accident Level'] == 'IV'].shape[0]
V_acc_cnt = industry_df[industry_df['Accident Level'] == 'V'].shape[0]
VI_acc_cnt = industry_df[industry_df['Accident Level'] == 'VI'].shape[0]

print(f'Accident Level - I count: {I_acc_cnt} i.e. {round(I_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Accident Level - II count: {II_acc_cnt} i.e. {round(II_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Accident Level - III count: {III_acc_cnt} i.e. {round(III_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Accident Level - IV count: {IV_acc_cnt} i.e. {round(IV_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Accident Level - V count: {V_acc_cnt} i.e. {round(V_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Accident Level - VI count: {VI_acc_cnt} i.e. {round(VI_acc_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Value Counts for `Potential Accident Level'); print('--'*40)

I_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'I'].shape[0]
II_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'II'].shape[0]
III_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'III'].shape[0]
IV_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'IV'].shape[0]
V_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'V'].shape[0]
VI_pot_acc_cnt = industry_df[industry_df['Potential Accident Level'] == 'VI'].shape[0]

print(f'Potential Accident Level - I count: {I_pot_acc_cnt} i.e. {round(I_pot_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Potential Accident Level - II count: {II_pot_acc_cnt} i.e. {round(II_pot_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Potential Accident Level - III count: {III_pot_acc_cnt} i.e. {round(III_pot_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Potential Accident Level - IV count: {IV_pot_acc_cnt} i.e. {round(IV_pot_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Potential Accident Level - V count: {V_pot_acc_cnt} i.e. {round(V_pot_acc_cnt/total_row_cnt*100, 0)}%')
print(f'Potential Accident Level - VI count: {VI_pot_acc_cnt} i.e. {round(VI_pot_acc_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Distributon of `Accident Level` & `Potential Accident Level` label'); print('--'*40)

ac_level_cnt = np.round(industry_df['Accident Level'].value_counts(normalize=True) * 100)
pot_ac_level_cnt = np.round(industry_df['Potential Accident Level'].value_counts(normalize=True) * 100, decimals=1)
ac_pot = pd.concat([ac_level_cnt, pot_ac_level_cnt], axis=1,sort=False).fillna(0).rename(columns={'Accident Level':'Accident', 'Potential Accident Level':'Potential'})
ac_pot = pd.melt(ac_pot.reset_index(), ['index']).rename(columns={'index':'Severity', 'variable':'Levels'})

hv.Bars(ac_pot, ['Severity', 'Levels'], 'value').opts(opts.Bars(title="Accident Levels Count", width=700, height=300,tools=['hover'],\
                                                                show_grid=True,xrotation=45, ylabel="Percentage", yformatter='%d%%'))

* The number of accidents decreases as the Accident Level increases.
* The number of accidents increases as the Potential Accident Level increases.

**Gender**

In [ ]:
print('--'*30); print('Value Counts for `Gender` label'); print('--'*30)

Male_cnt = industry_df[industry_df['Gender'] == 'Male'].shape[0]
Female_cnt = industry_df[industry_df['Gender'] == 'Female'].shape[0]

print(f'Male count: {Male_cnt} i.e. {round(Male_cnt/total_row_cnt*100, 0)}%')
print(f'Female count: {Female_cnt} i.e. {round(Female_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Distributon of `Gender` label'); print('--'*30)

gender_cnt = np.round(industry_df['Gender'].value_counts(normalize=True) * 100)

hv.Bars(gender_cnt).opts(title="Gender Count", color="#8888ff", xlabel="Gender", ylabel="Percentage", yformatter='%d%%')\
                .opts(opts.Bars(width=500, height=300,tools=['hover'],show_grid=True))

* There are more men working in this industry as compared to women.

**Employee type**

In [ ]:
print('--'*30); print('Value Counts for `Employee type` label'); print('--'*30)

third_party_cnt = industry_df[industry_df['Employee type'] == 'Third Party'].shape[0]
emp_cnt = industry_df[industry_df['Employee type'] == 'Employee'].shape[0]
third_rem_cnt = industry_df[industry_df['Employee type'] == 'Third Party (Remote)'].shape[0]

print(f'Third Party count: {third_party_cnt} i.e. {round(third_party_cnt/total_row_cnt*100, 0)}%')
print(f'Employee count: {emp_cnt} i.e. {round(emp_cnt/total_row_cnt*100, 0)}%')
print(f'Third Party (Remote) count: {third_rem_cnt} i.e. {round(third_rem_cnt/total_row_cnt*100, 0)}%')

print('--'*30); print('Distributon of `Employee type` label'); print('--'*30)

emp_type_cnt = np.round(industry_df['Employee type'].value_counts(normalize=True) * 100)

hv.Bars(emp_type_cnt).opts(title="Employee type Count", color="#8888ff", xlabel="Employee Type", ylabel="Percentage", yformatter='%d%%')\
                .opts(opts.Bars(width=500, height=300,tools=['hover'],show_grid=True))

* 44% Third party empoyees working in this industry.
* 43% own empoyees working in this industry.
* 13% Third party(Remote) empoyees working in this industry.

**Critical Risk**

In [ ]:
cr_risk_cnt = np.round(industry_df['Critical Risk'].value_counts(normalize=True) * 100)

hv.Bars(cr_risk_cnt[::-1]).opts(title="Critical Risk Count", color="#8888ff", xlabel="Critical Risks", ylabel="Percentage", xformatter='%d%%')\
                .opts(opts.Bars(width=600, height=600,tools=['hover'],show_grid=True,invert_axes=True))

* Because most part of the Critical Risks are classified as 'Others', it is thought that there are too many risks to classify precisely.

* And it is also thought that it takes so much time to analyze risks and reasons why the accidents occur.

**Calendar**

In [ ]:
year_cnt = np.round(industry_df['Year'].value_counts(normalize=True,sort=False) * 100)
year = hv.Bars(year_cnt).opts(title="Year Count", color="yellow", xlabel="Years")

month_cnt = np.round(industry_df['Month'].value_counts(normalize=True,sort=False) * 100)
month = hv.Bars(month_cnt).opts(title="Month Count", color="#8888ff", xlabel="Months") * hv.Curve(month_cnt).opts(color='red', line_width=3)

(year + month).opts(opts.Bars(width=400, height=300,tools=['hover'],show_grid=True, ylabel="Percentage", yformatter='%d%%')).cols(2)

* Accidents are recorded from 1st Jan 2016 to 9th July 2017 in every month, there are high number of accidents in 2016 and less in 2017.
* Number of accidents are high in beginning of the year and it keeps decreasing later.

In [ ]:
day_cnt = np.round(industry_df['Day'].value_counts(normalize=True,sort=False) * 100)
hv.Bars(day_cnt).opts(title="Day Count", color="#8888ff", xlabel="Days") * hv.Curve(day_cnt).opts(width=500, height=300, color='red', line_width=3)

* Number of accidents are very high in particular days like 4, 8 and 16 in every month.

In [ ]:
weekday_cnt = pd.DataFrame(np.round(industry_df['Weekday'].value_counts(normalize=True,sort=False) * 100))
weekday_cnt['week_num'] = [['aMonday', 'bTuesday', 'cWednesday', 'dThursday', 'eFriday', 'fSaturday', 'gSunday'].index(i) for i in weekday_cnt.index]
weekday_cnt.sort_values('week_num', inplace=True)

hv.Bars((weekday_cnt.index, weekday_cnt.Weekday)).opts(title="Weekday Count", color="#8888ff", xlabel="Weekdays") * hv.Curve(weekday_cnt['Weekday']).opts(width=500, height=300, color='red', line_width=3)
#(day + weekday).opts(opts.Bars(width=400, height=300,tools=['hover'],show_grid=True, ylabel="Percentage", yformatter='%d%%')).cols(2)

* Number of accidents increased during the middle of the week and declined since the middle of th week.

[Table of Contents](#table-of-contents)

<a id="bivariate-analysis"></a>
#### Bivariate Analysis and Hypothesis testing

##### a. Industry Sector by Countries - Is the distribution of industry sector different significantly in differ countries or not?

In [ ]:
# Check the proportion of Industry sector in different countries
indsec_cntry_table = pd.crosstab(index = industry_df['Industry Sector'], columns = industry_df['Country'])
indsec_cntry_table.plot(kind = 'bar', figsize=(8,8), stacked = True)
plt.title("Proportion of Industry Sector in different countries")
plt.show()

**Observations**

* Metals and Mining industry sector plants are not available in Country_03.
* Distribution of industry sector differ significantly in each country. But let's check the proportion of metals, mining and others sector in Country_01 and is that difference is statistically significant?

###### 1. State the H0 and Ha

###### Ho = The proportions of industry sector is not differ in different countries
###### Ha = The proportions of industry sector is differ in different countries

###### 2. Decide the significance level: alpha = 0.05

###### 3. Identify the test-statistic: Z-test of proportions

###### 4. Calculate the p_value using test-statistic

In [ ]:
mining_country1 = industry_df[industry_df['Industry Sector'] == 'Mining']['Country'].value_counts()[0]
mining_country2 = industry_df[industry_df['Industry Sector'] == 'Mining']['Country'].value_counts()[1]

metals_country1 = industry_df[industry_df['Industry Sector'] == 'Metals']['Country'].value_counts()[1]
metals_country2 = industry_df[industry_df['Industry Sector'] == 'Metals']['Country'].value_counts()[0]

others_country1 = industry_df[industry_df['Industry Sector'] == 'Others']['Country'].value_counts()[2]
others_country2 = industry_df[industry_df['Industry Sector'] == 'Others']['Country'].value_counts()[1]
others_country3 = industry_df[industry_df['Industry Sector'] == 'Others']['Country'].value_counts()[0]

print([mining_country1, metals_country1, others_country1], [country_01_cnt])
print(f'Proportions of mining, metals, others in country_01 = {round(200/248,2)}%, {round(46/248,2)}%, {round(2/248,2)}% respectively')

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Z-test proportions: More than 2 samples not implemented yet, hence I am passing two elements
t_statistic, p_value = proportions_ztest([mining_country1, metals_country1], [country_01_cnt])

print("Mining and Metals t_statistic", t_statistic)
print("Mining and Metals p_value", p_value)

t_statistic, p_value = proportions_ztest([mining_country1, others_country1], [country_01_cnt])

print("Mining and Others t_statistic", t_statistic)
print("Mining and Others p_value", p_value)

###### 5. Decide to Reject or Accept Null Hypothesis

In [ ]:
reject_null = False
if p_value < 0.05:
    reject_null = True 
else: 
    reject_null = False
    
print("reject null? : " + str(reject_null))

* Hence we reject Null Hypothesis, we have enough (95%) evidence to prove that, the mining sector in country 1 is differ from metals sector)

* Hence we reject Null Hypothesis, we have enough (95%) evidence to prove that, the mining sector in country 1 is differ from others sector)

##### b. Employee type by Gender - Is the distribution of employee type differ significantly in different genders?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)
em_gen = industry_df.groupby(['Gender','Employee type'])['Employee type'].count().unstack().apply(f, axis=1)

hv.Bars(pd.melt(em_gen.reset_index(), ['Gender']), ['Gender','Employee type'], 'value').opts(opts.Bars(title="Employee type by Gender Count", width=800, height=300,tools=['hover'],\
                                                                show_grid=True,xrotation=0, ylabel="Percentage", yformatter='%d%%'))

**Observations**

* Proportion of third party employees in each gender is equal.
* Proportion of third party(remote) employees in each gender is not equal.
* Proportion of own employees in each gender is not equal. But let's check is that difference is statistically significant?

###### 1. State the H0 and Ha

###### Ho = The proportions of own employees in each gender is equal.
###### Ha = The proportions of own employees in each gender is not equal.

###### 2. Decide the significance level: alpha = 0.05

###### 3. Identify the test-statistic: Z-test of proportions

###### 4. Calculate the p_value using test-statistic

In [ ]:
male_emp = industry_df[industry_df['Employee type'] == 'Employee'].Gender.value_counts()[0]
female_emp = industry_df[industry_df['Employee type'] == 'Employee'].Gender.value_counts()[1]

print([male_emp, female_emp], [Male_cnt, Female_cnt])
print(f'Proportion of own employee types in male, female = {round(170/396,2)}%, {round(8/22,2)}% respectively')

In [ ]:
t_statistic, p_value = proportions_ztest([male_emp, female_emp], [Male_cnt, Female_cnt])

print("t_statistic", t_statistic)
print("p_value", p_value)

###### 5. Decide to Reject or Accept Null Hypothesis

In [ ]:
reject_null = False
if p_value < 0.05:
    reject_null = True 
else: 
    reject_null = False
    
print("reject null? : " + str(reject_null))

Hence we fail to reject Null Hypothesis, we have enough (95%) evidence to prove that, the proportion of own employees in each gender is equal.

##### c. Industry Sector by Gender - Is the distribution of industry sector differ significantly in different genders?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)
em_gen = industry_df.groupby(['Gender','Industry Sector'])['Industry Sector'].count().unstack().apply(f, axis=1)

hv.Bars(pd.melt(em_gen.reset_index(), ['Gender']), ['Gender','Industry Sector'], 'value').opts(opts.Bars(title="Industry Sector by Gender Count", width=800, height=300,tools=['hover'],\
                                                                show_grid=True,xrotation=0, ylabel="Percentage", yformatter='%d%%'))

**Observations**

* Proportion of Metals sector employees in each gender is not equal.
* Proportion of Mining sector employees in each gender is not equal.
* Proportion of Others sector employees in each gender is not equal.

##### d. Accident Levels by Gender - Is the distribution of accident levels and potential accident levels differ significantly in different genders?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)

ac_gen = industry_df.groupby(['Gender','Accident Level'])['Accident Level'].count().unstack().apply(f, axis=1)
ac = hv.Bars(pd.melt(ac_gen.reset_index(), ['Gender']), ['Gender','Accident Level'], 'value').opts(opts.Bars(title="Accident Level by Gender Count"))

pot_ac_gen = industry_df.groupby(['Gender','Potential Accident Level'])['Potential Accident Level'].count().unstack().apply(f, axis=1)
pot_ac = hv.Bars(pd.melt(pot_ac_gen.reset_index(), ['Gender']), ['Gender','Potential Accident Level'], 'value').opts(opts.Bars(title="Potential Accident Level by Gender Count"))

(ac + pot_ac).opts(opts.Bars(width=400, height=300,tools=['hover'],show_grid=True,xrotation=0, ylabel="Percentage", yformatter='%d%%'))

**Observations**

* Proportion of accident levels in each gender is not equal and males have a higher accident levels than females.
* There are many low risks at general accident level, but many high risks at potential accident level.

##### e. Accident Levels by Employee type - Is the distribution of accident levels and potential accident levels differ significantly in different employee types?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)

ac_em = industry_df.groupby(['Employee type','Accident Level'])['Accident Level'].count().unstack().apply(f, axis=1)
ac = hv.Bars(pd.melt(ac_em.reset_index(), ['Employee type']), ['Employee type','Accident Level'], 'value').opts(opts.Bars(title="Accident Level by Employee type Count"))

pot_ac_em = industry_df.groupby(['Employee type','Potential Accident Level'])['Potential Accident Level'].count().unstack().apply(f, axis=1)
pot_ac = hv.Bars(pd.melt(pot_ac_em.reset_index(), ['Employee type']), ['Employee type','Potential Accident Level'], 'value').opts(opts.Bars(title="Potential Accident Level by Employee type Count"))

(ac + pot_ac).opts(opts.Bars(width=400, height=300,tools=['hover'],show_grid=True,xrotation=0, ylabel="Percentage", yformatter='%d%%',fontsize={'title':9}))

**Observations**

* For both accident levels, the incidence of Employee is higher at low accident levels, but the incidence of Third parties seems to be slightly higher at high accident levels.

##### f. Accident Levels by Month - Is the distribution of accident levels and potential accident levels differ significantly in different months?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)

ac_mo = industry_df.groupby(['Month','Accident Level'])['Accident Level'].count().unstack().apply(f, axis=1).fillna(0)
ac = hv.Curve(ac_mo['I'], label='I') * hv.Curve(ac_mo['II'], label='II') * hv.Curve(ac_mo['III'], label='III') * hv.Curve(ac_mo['IV'], label='IV') * hv.Curve(ac_mo['V'], label='V')\
        .opts(opts.Curve(title="Accident Level by Month Count"))

pot_ac_mo = industry_df.groupby(['Month','Potential Accident Level'])['Potential Accident Level'].count().unstack().apply(f, axis=1).fillna(0)
pot_ac = hv.Curve(pot_ac_mo['I'], label='I') * hv.Curve(pot_ac_mo['II'], label='II') * hv.Curve(pot_ac_mo['III'], label='III') * hv.Curve(pot_ac_mo['IV'], label='IV')\
        * hv.Curve(pot_ac_mo['V'], label='V') * hv.Curve(pot_ac_mo['VI'], label='VI').opts(opts.Curve(title="Potential Accident Level by Month Count"))
        
(ac+pot_ac).opts(opts.Curve(width=800, height=300,tools=['hover'],show_grid=True, ylabel="Percentage", yformatter='%d%%')).cols(1)

**Observations**

* Both of the two accident level have the tendency that non-severe levels decreased throughout the year, but severe levels did not changed much, and some of these levels increased slightly in the second half of the year.

##### g. Accident Levels by Weekday - Is the distribution of accident levels and potential accident levels differ significantly in different weekday?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)

ac_weekday = industry_df.groupby(['Weekday','Accident Level'])['Accident Level'].count().unstack().apply(f, axis=1).fillna(0)
ac_weekday['week_num'] = [['aMonday', 'bTuesday', 'cWednesday', 'dThursday', 'eFriday', 'fSaturday', 'gSunday'].index(i) for i in ac_weekday.index]
ac_weekday.sort_values('week_num', inplace=True)
ac_weekday.drop('week_num', axis=1, inplace=True)
ac = hv.Curve(ac_weekday['I'], label='I') * hv.Curve(ac_weekday['II'], label='II') * hv.Curve(ac_weekday['III'], label='III') * hv.Curve(ac_weekday['IV'], label='IV') * hv.Curve(ac_weekday['V'], label='V')\
        .opts(opts.Curve(title="Accident Level by Weekday Count"))

pot_ac_weekday = industry_df.groupby(['Weekday','Potential Accident Level'])['Potential Accident Level'].count().unstack().apply(f, axis=0).fillna(0)
pot_ac_weekday['week_num'] = [['aMonday', 'bTuesday', 'cWednesday', 'dThursday', 'eFriday', 'fSaturday', 'gSunday'].index(i) for i in pot_ac_weekday.index]
pot_ac_weekday.sort_values('week_num', inplace=True)
pot_ac_weekday.drop('week_num', axis=1, inplace=True)
pot_ac = hv.Curve(pot_ac_weekday['I'], label='I') * hv.Curve(pot_ac_weekday['II'], label='II') * hv.Curve(pot_ac_weekday['III'], label='III') * hv.Curve(pot_ac_weekday['IV'], label='IV')\
        * hv.Curve(pot_ac_weekday['V'], label='V') * hv.Curve(pot_ac_weekday['VI'], label='VI').opts(opts.Curve(title="Potential Accident Level by Weekday Count"))

(ac+pot_ac).opts(opts.Curve(width=800, height=300,tools=['hover'],show_grid=True, ylabel="Percentage", yformatter='%d%%')).cols(1)

**Observations**

* Both of the two accident level is thought that non-severe levels decreased in the first and the last of the week, but severe levels did not changed much.

##### h. Accident Levels by Seasons - Is the distribution of accident levels and potential accident levels differ significantly in different seasons?

In [ ]:
f = lambda x : np.round(x/x.sum() * 100)
ac_season = industry_df.groupby(['Season','Accident Level'])['Accident Level'].count().unstack().apply(f, axis=1).fillna(0)
ac_season['season_num'] = [['dSpring', 'aSummer', 'bAutumn', 'cWinter'].index(i) for i in ac_season.index]
ac_season.sort_values('season_num', inplace=True)
ac_season.drop('season_num', axis=1, inplace=True)
ac = hv.Curve(ac_season['I'], label='I') * hv.Curve(ac_season['II'], label='II') * hv.Curve(ac_season['III'], label='III') * hv.Curve(ac_season['IV'], label='IV') * hv.Curve(ac_season['V'], label='V')\
        .opts(opts.Curve(title="Accident Level by Season Count"))

pot_ac_season = industry_df.groupby(['Season','Potential Accident Level'])['Potential Accident Level'].count().unstack().apply(f, axis=0).fillna(0)
pot_ac_season['season_num'] = [['dSpring', 'aSummer', 'bAutumn', 'cWinter'].index(i) for i in pot_ac_season.index]
pot_ac_season.sort_values('season_num', inplace=True)
pot_ac_season.drop('season_num', axis=1, inplace=True)
pot_ac = hv.Curve(pot_ac_season['I'], label='I') * hv.Curve(pot_ac_season['II'], label='II') * hv.Curve(pot_ac_season['III'], label='III') * hv.Curve(pot_ac_season['IV'], label='IV')\
        * hv.Curve(pot_ac_season['V'], label='V') * hv.Curve(pot_ac_season['VI'], label='VI').opts(opts.Curve(title="Potential Accident Level by Season Count"))

(ac+pot_ac).opts(opts.Curve(width=800, height=300,tools=['hover'],show_grid=True, ylabel="Percentage", yformatter='%d%%')).cols(1)

**Observations**

* Both of the two accident level have the tendency that non-severe levels decreased throughout the year, but severe levels did not changed much, and some of these levels increased slightly in the second half of the year.

#### Study Summary Statistics

In [ ]:
# Summary statistics
industry_df.drop(columns='Description').describe(exclude=[np.number]).T

#### Study Correlation

In [ ]:
# Check the Correlation
industry_df.corr()

**Observations**

* WeekofYear featuer is having very high positive correlation with Month feature.

## EDA Summary:



**Local**
* Highest manufacturing plants are located in Local_03 city and lowest in Local_09 city.

**Country**
* Percentage(%) of accidents occurred in respective countries: 59% in Country_01, 31% in Country_02 and 10% in Country_03.

**Industry Sector**
* Percentage(%) of manufacturing plants belongs to respective sectors: 57% to Mining sector, 32% to Metals sector and 11% to Others sector.

**Country + Industry Sector**
* Metals and Mining industry sector plants are not available in Country_03.
* Distribution of industry sector differ significantly in each country.

**Accident Levels**
* The number of accidents decreases as the Accident Level increases and increases as the Potential Accident Level increases.

**Gender**
* There are more men working in this industry as compared to women.

**Employee type**
* 44% Third party empoyees, 43% own empoyees and 13% Third party(Remote) empoyees working in this industry.

**Gender + Employee type**
* Proportion of third party employees in each gender is equal, third party(remote) employees in each gender is not equal and 
own employees in each gender is not equal.

**Gender + Industry Sector**
* Proportion of Metals, Mining and Others sector employees in each gender is not equal

**Gender + Accident Levels**
* Males have a higher accident levels than females.
* There are many low risks at general accident level, but many high risks at potential accident level.

**Accident Levels + Employee type**
* For both accident levels, the incidence of Employee is higher at low accident levels, but the incidence of Third parties seems to be 
slightly higher at high accident levels.

**Accident Levels + Calendar**
* Accidents are recorded from 1st Jan 2016 to 9th July 2017 in every month, there are high number of accidents in 2016 and less in 2017.
* Number of accidents are high in beginning of the year and it keeps decreasing later.
* Number of accidents are very high in particular days like 4, 8 and 16 in every month.
* Number of accidents increased during the middle of the week and declined since the middle of th week.

* Both of the two accident level have the tendency that non-severe levels decreased throughout the year, but severe levels did not changed much, 
and some of these levels increased slightly in the second half of the year.
* Both of the two accident level is thought that non-severe levels decreased in the first and the last of the week, but severe levels did not 
changed much.

**Critical Risk**
* Most of the critical risks are classified as Others.

[Table of Contents](#table-of-contents)

<a id="nlp-analysis"></a>
## 7. NLP Analysis

In [ ]:
# Checking 5 random Description and accident_levels from the data
print('--'*35); print('Checking 5 random Descriptions and accident_levels from the data'); print('--'*35)
rands = random.sample(range(1, industry_df.shape[0]), 5)
descriptions, accident_levels = list(industry_df.loc[rands, 'Description']), list(industry_df.loc[rands, 'Accident Level'])

_ = [print(f'Description: {description}\naccident_level: {acclevel}\n') for description, acclevel in zip(descriptions, accident_levels)]

In [ ]:
# Checking 5 random Descriptions and accident_levels from the data where the length of headline is > 100
print('--'*55); print('Checking 5 random Descriptions and accident_levels from the data where the length of Description is > 100'); print('--'*55)
indexes = list(industry_df.loc[industry_df['Description'].str.len() > 100, 'Description'].index)
rands = random.sample(indexes, 5)
descriptions, accident_levels = list(industry_df.loc[rands, 'Description']), list(industry_df.loc[rands, 'Accident Level'])

_ = [print(f'Description: {description}\naccident_level: {acclevel}\n') for description, acclevel in zip(descriptions, accident_levels)]

print('--'*40); print('Distributon of accident_level where the length of Description is > 100'); print('--'*40)
_ = industry_df.loc[indexes, 'Accident Level'].value_counts().plot(kind = 'pie', autopct = '%.0f%%', labels = ['I', 'II', 'III', 'IV', 'V'], figsize = (10, 6))


**Observations**

- 74% of data where accident description > 100 is captured in low accident level. 
- Based on some random headlines seen above, it appears that the data is mostly lower-cased. Pre-processing such as removing punctuations and lemmatization can be used.
- There are few alphanumeric characters like 042-TC-06, Nv. 3370, CX 212 captured in description where removing these characters might help.
- There are digits in the description for e.g. level 326, Dumper 01 where removing the digits wouldn't help.

In [ ]:
# Checking 5 random Descriptions and pot_accident_levels from the data where the length of headline is > 100
print('--'*60); print('Checking 5 random Descriptions and pot_accident_levels from the data where the length of Description is > 100'); print('--'*60)
indexes = list(industry_df.loc[industry_df['Description'].str.len() > 100, 'Description'].index)
rands = random.sample(indexes, 5)
descriptions, pot_accident_levels = list(industry_df.loc[rands, 'Description']), list(industry_df.loc[rands, 'Potential Accident Level'])

_ = [print(f'Description: {descriptin}\npot_accident_level: {pot_acclevel}\n') for descriptin, pot_acclevel in zip(descriptions, pot_accident_levels)]

print('--'*40); print('Distributon of pot_accident_level where the length of Description is > 100'); print('--'*40)
_ = industry_df.loc[indexes, 'Potential Accident Level'].value_counts().plot(kind = 'pie', autopct = '%.0f%%', labels = ['IV', 'III', 'II', 'I', 'V', 'VI'], figsize = (10, 6))


**Observations**

- 34% of data where accident description > 100 is captured in high medium potential accident level.
- 25% of data where accident description > 100 is captured in medium potential accident level.
- 23% of data where accident description > 100 is captured in low potential accident level. 
- Based on some random headlines seen above, it appears that the data is mostly lower-cased. Pre-processing such as removing punctuations and lemmatization can be used.
- There are few alphanumeric characters like AFO-755 captured in description where removing these characters might help.
- There are digits in the description for e.g. ditch 3570, 0.50 cm deep, 30 kg where removing the digits wouldn't help.

[Table of Contents](#table-of-contents)

<a id="nlp-pre-processing"></a>
## 8. NLP Pre-processing

Few of the NLP pre-processing steps taken before applying model on the data

- Converting to lower case, avoid any capital cases
- Converting apostrophe to the standard lexicons
- Removing punctuations
- Lemmatization
- Removing stop words

In [ ]:
# Text preprocessing and stopwords
from text_preprocess_py import * #(custom module)

In [ ]:
print('--'*30); print('Converting description to lower case')
industry_df['Cleaned_Description'] = industry_df['Description'].apply(lambda x : x.lower())

print('Replacing apostrophes to the standard lexicons')
industry_df['Cleaned_Description'] = industry_df['Cleaned_Description'].apply(lambda x : replace_words(x))

print('Removing punctuations')
industry_df['Cleaned_Description'] = industry_df['Cleaned_Description'].apply(lambda x: remove_punctuation(x))

print('Applying Lemmatizer')
industry_df['Cleaned_Description'] = industry_df['Cleaned_Description'].apply(lambda x: lemmatize(x))

print('Removing multiple spaces between words')
industry_df['Cleaned_Description'] = industry_df['Cleaned_Description'].apply(lambda x: re.sub(' +', ' ', x))

print('Removing stop words')
industry_df['Cleaned_Description'] = industry_df['Cleaned_Description'].apply(lambda x: remove_stopwords(x))

print('--'*30)

#### Get the Length of each line and find the maximum length

As different lines are of different length. We need to pad the our sequences using the max length.

In [ ]:
print('--'*45); print('Get the length of each line, find the maximum length and print the maximum length line'); 
print('Length of line ranges from 64 to 672.'); print('--'*45)

# Get length of each line
industry_df['line_length'] = industry_df['Cleaned_Description'].str.len()

print('Minimum line length: {}'.format(industry_df['line_length'].min()))
print('Maximum line length: {}'.format(industry_df['line_length'].max()))
print('Line with maximum length: {}'.format(industry_df[industry_df['line_length'] == industry_df['line_length'].max()]['Cleaned_Description'].values[0]))

In [ ]:
print('--'*45); print('Get the number of words, find the maximum number of words and print the maximum number of words'); 
print('Number of words ranges from 10 to 98.'); print('--'*45)

# Get length of each line
industry_df['nb_words'] = industry_df['Cleaned_Description'].apply(lambda x: len(x.split(' ')))

print('Minimum number of words: {}'.format(industry_df['nb_words'].min()))
print('Maximum number of words: {}'.format(industry_df['nb_words'].max()))
print('Line with maximum number of words: {}'.format(industry_df[industry_df['nb_words'] == industry_df['nb_words'].max()]['Cleaned_Description'].values[0]))

<a id="wordcloud"></a>
#### WordCloud

In [ ]:
wordcloud = WordCloud(width = 1500, height = 800, random_state=0, background_color='black', colormap='rainbow', \
                      min_font_size=5, max_words=300, collocations=False).generate(" ".join(industry_df['Cleaned_Description'].values))
plt.figure(figsize=(15,10))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

**Observations**

There are many body-related, employee related, movement-related, equipment-related and accident-related words.

* Body-related: left, right, hand, finger, face, foot and glove
* Employee-related: employee, operator, collaborator, assistant, worker and mechanic
* Movement-related: fall, hit, lift and slip
* Equipment-related: equipment, pump, meter, drill, truck and tube
* Accident-related: accident, activity, safety, injury, causing


#### NLP text summary statistics

In [ ]:
print('--'*30); print('Five point summary for number of words')
display(industry_df['nb_words'].describe().round(0).astype(int)); 

print('99% quantilie: {}'.format(industry_df['nb_words'].quantile(0.99)));print('--'*30)

## NLP Pre-processing Summary:

- 74% of data where accident description > 100 is captured in low accident level.
- 34% of data where accident description > 100 is captured in high medium potential accident level.
- 25% of data where accident description > 100 is captured in medium potential accident level.
- 23% of data where accident description > 100 is captured in low potential accident level. 
- Few of the NLP pre-processing steps taken before applying model on the data

  - Converting to lower case, avoid any capital cases
  - Converting apostrophe to the standard lexicons
  - Removing punctuations
  - Lemmatization
  - Removing stop words

- After pre-processing steps:
  - Minimum line length: 64
  - Maximum line length: 672
  - Minimum number of words: 10
  - Maximum number of words: 98

[Table of Contents](#table-of-contents)

<a id="feature-engineering"></a>
## 9. Feature Engineering

#### Variable Creation - Word2Vec Embeddings

In [ ]:
from gensim.models import Word2Vec
# define training data
sentences = industry_df['Cleaned_Description']

# train model
model = Word2Vec(sentences, min_count=1)

# summarize the loaded model
print(model)

# summarize vocabulary
words = list(model.wv.index_to_key)
print(words)

# save model
model.save('model.bin')

# load model
new_model = Word2Vec.load('model.bin')
print(new_model)

#### Variable Creation - Glove Word Embeddings

In [ ]:
embeddings_index = {}
EMBEDDING_FILE = '../input/glove6b200d/glove.6B.200d.txt'
f = open(EMBEDDING_FILE)
for line in tqdm(f):
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

In [ ]:
# this function creates a normalized vector for the whole sentence
def sent2vec(s):
    words = str(s).lower()
    words = word_tokenize(words)
    words = [w for w in words if not w in stop_words]
    words = [w for w in words if w.isalpha()]
    M = []
    for w in words:
        try:
            M.append(embeddings_index[w])
        except:
            continue
    M = np.array(M)
    v = M.sum(axis=0)
    if type(v) != np.ndarray:
        return np.zeros(300)
    return v / np.sqrt((v ** 2).sum())

In [ ]:
# create sentence GLOVE embeddings vectors using the above function for training and validation set
ind_glove_df = [sent2vec(x) for x in tqdm(industry_df['Cleaned_Description'])]

In [ ]:
ind_glove_df[0]

#### Variable Creation - TFIDF Features

In [ ]:
ind_tfidf_df = pd.DataFrame()
for i in [1,2,3]:
    vec_tfidf = TfidfVectorizer(max_features=10, norm='l2', stop_words='english', lowercase=True, use_idf=True, ngram_range=(i,i))
    X = vec_tfidf.fit_transform(industry_df['Cleaned_Description']).toarray()
    tfs = pd.DataFrame(X, columns=["TFIDF_" + n for n in vec_tfidf.get_feature_names()])
    ind_tfidf_df = pd.concat([ind_tfidf_df.reset_index(drop=True), tfs.reset_index(drop=True)], axis=1)

ind_tfidf_df.head(3)

#### Variable Creation - Label Encoding

In [ ]:
# To replace white space everywhere in Employee type
industry_df['Employee type'] = industry_df['Employee type'].str.replace(' ', '_')
industry_df['Employee type'].value_counts()

In [ ]:
# To replace white space everywhere in Critical Risk
industry_df['Critical Risk'] = industry_df['Critical Risk'].str.replace('\n', '').str.replace(' ', '_')
industry_df['Critical Risk'].value_counts().head()

In [ ]:
# Create Industry DataFrame
ind_featenc_df = pd.DataFrame()

# Label encoding
industry_df['Season'] = industry_df['Season'].replace('Summer', 'aSummer').replace('Autumn', 'bAutumn').replace('Winter', 'cWinter').replace('Spring', 'dSpring')
ind_featenc_df['Season'] = LabelEncoder().fit_transform(industry_df['Season']).astype(np.int8)

industry_df['Weekday'] = industry_df['Weekday'].replace('Monday', 'aMonday').replace('Tuesday', 'bTuesday').replace('Wednesday', 'cWednesday').replace('Thursday', 'dThursday').replace('Friday', 'eFriday').replace('Saturday', 'fSaturday').replace('Sunday', 'gSunday')
ind_featenc_df['Weekday'] = LabelEncoder().fit_transform(industry_df['Weekday']).astype(np.int8)

ind_featenc_df['Accident Level'] = LabelEncoder().fit_transform(industry_df['Accident Level']).astype(np.int8)
ind_featenc_df['Potential Accident Level'] = LabelEncoder().fit_transform(industry_df['Potential Accident Level']).astype(np.int8)

In [ ]:
# convert integers to dummy variables (i.e. one hot encoded)
dummy_y = np_utils.to_categorical(ind_featenc_df['Accident Level'])
dummy_y

In [ ]:
# Dummy variables encoding
Country_dummies = pd.get_dummies(industry_df['Country'], columns=["Country"], drop_first=True)
Local_dummies = pd.get_dummies(industry_df['Local'], columns=["Local"], drop_first=True)
Gender_dummies = pd.get_dummies(industry_df['Gender'], columns=["Gender"], drop_first=True)
IS_dummies = pd.get_dummies(industry_df['Industry Sector'], columns=['Industry Sector'], prefix='IS', drop_first=True)
EmpType_dummies = pd.get_dummies(industry_df['Employee type'], columns=['Employee type'], prefix='EmpType', drop_first=True)
CR_dummies = pd.get_dummies(industry_df['Critical Risk'], columns=['Critical Risk'], prefix='CR', drop_first=True)

# Merge the above dataframe with the original dataframe ind_feat_df
ind_featenc_df = ind_featenc_df.join(Country_dummies.reset_index(drop=True)).join(Local_dummies.reset_index(drop=True)).join(Gender_dummies.reset_index(drop=True)).join(IS_dummies.reset_index(drop=True)).join(EmpType_dummies.reset_index(drop=True)).join(CR_dummies.reset_index(drop=True))

ind_featenc_df = industry_df[['Year','Month','Day','WeekofYear']].reset_index(drop=True).join(ind_featenc_df.reset_index(drop=True))

ind_featenc_df.head(3)

In [ ]:
# Check NaN values
np.any(np.isnan(ind_featenc_df))

#### Combine Glove and Encoded Features

In [ ]:
# Consider only top 30 GLOVE features
ind_feat_df = ind_featenc_df.join(pd.DataFrame(ind_glove_df).iloc[:,0:30].reset_index(drop=True))

In [ ]:
ind_feat_df.head(3)

#### Combine TFIDF and Encoded Features

In [ ]:
# Consider only top 30 GLOVE features
ind_feat_df = ind_featenc_df.join(ind_tfidf_df.reset_index(drop=True))

In [ ]:
ind_feat_df.head(3)

#### Sampling Techniques - Create Training and Test Set

In [ ]:
X = ind_feat_df.drop(['Accident Level','Potential Accident Level'], axis = 1) # Considering all Predictors
y = ind_feat_df['Accident Level']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 1, stratify = y)

In [ ]:
X_train, X_test, y_train_dummy, y_test_dummy = train_test_split(X, dummy_y, test_size = 0.20, random_state = 1, stratify = y)

In [ ]:
print('X_train shape : ({0},{1})'.format(X_train.shape[0], X_train.shape[1]))
print('y_train shape : ({0},)'.format(y_train.shape[0]))
print('X_test shape : ({0},{1})'.format(X_test.shape[0], X_test.shape[1]))
print('y_test shape : ({0},)'.format(y_test.shape[0]))

#### Resampling Techniques — Oversample minority class

In [ ]:
# Display old accident level counts
ind_feat_df['Accident Level'].value_counts()

In [ ]:
# Concatenate our training data back together
X_up = pd.concat([X_train, y_train], axis=1)

# Get the majority and minority class
acclevel_0_majority = X_up[X_up['Accident Level'] == 0]
acclevel_1_minority = X_up[X_up['Accident Level'] == 1]
acclevel_2_minority = X_up[X_up['Accident Level'] == 2]
acclevel_3_minority = X_up[X_up['Accident Level'] == 3]
acclevel_4_minority = X_up[X_up['Accident Level'] == 4]

# Upsample Level1 minority class
acclevel_1_minority_upsampled = resample(acclevel_1_minority,
                                replace = True, # sample with replacement
                                n_samples = len(acclevel_0_majority), # to match majority class
                                random_state = 1)

# Upsample Level2 minority class
acclevel_2_minority_upsampled = resample(acclevel_2_minority,
                                replace = True, # sample with replacement
                                n_samples = len(acclevel_0_majority), # to match majority class
                                random_state = 1)

# Upsample Level3 minority class
acclevel_3_minority_upsampled = resample(acclevel_3_minority,
                                replace = True, # sample with replacement
                                n_samples = len(acclevel_0_majority), # to match majority class
                                random_state = 1)

# Upsample Level4 minority class
acclevel_4_minority_upsampled = resample(acclevel_4_minority,
                                replace = True, # sample with replacement
                                n_samples = len(acclevel_0_majority), # to match majority class
                                random_state = 1)

In [ ]:
# Combine majority class with upsampled minority classes
df_upsampled = pd.concat([acclevel_0_majority, acclevel_1_minority_upsampled, acclevel_2_minority_upsampled, acclevel_3_minority_upsampled, 
                          acclevel_4_minority_upsampled])

In [ ]:
# Display new accident level counts
df_upsampled['Accident Level'].value_counts()

In [ ]:
# Separate input features and target
X_train_up = df_upsampled.drop(['Accident Level'], axis = 1) # Considering all Predictors
y_train_up = df_upsampled['Accident Level']

#### SMOTE - Generate synthetic samples - upsample smaller class

In [ ]:
sm = SMOTE(random_state=1)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)
df_smote = pd.concat([pd.DataFrame(X_train_smote), pd.DataFrame(y_train_smote)], axis=1)
df_smote.columns = ['Year', 'Month', 'Day',
        'WeekofYear', 'Season', 'Weekday',
       'Country_02', 'Country_03', 'Local_02', 'Local_03', 'Local_04',
       'Local_05', 'Local_06', 'Local_07', 'Local_08', 'Local_09', 'Local_10',
       'Local_11', 'Local_12', 'Male', 'IS_Mining', 'IS_Others',
       'EmpType_Third_Party', 'EmpType_Third_Party_(Remote)',
       'CR_Blocking_and_isolation_of_energies', 'CR_Burn',
       'CR_Chemical_substances', 'CR_Confined_space', 'CR_Cut',
       'CR_Electrical_Shock', 'CR_Electrical_installation', 'CR_Fall',
       'CR_Fall_prevention', 'CR_Fall_prevention_(same_level)',
       'CR_Individual_protection_equipment', 'CR_Liquid_Metal',
       'CR_Machine_Protection', 'CR_Manual_Tools', 'CR_Not_applicable',
       'CR_Others', 'CR_Plates', 'CR_Poll', 'CR_Power_lock', 'CR_Pressed',
       'CR_Pressurized_Systems',
       'CR_Pressurized_Systems_/_Chemical_Substances', 'CR_Projection',
       'CR_Projection/Burning', 'CR_Projection/Choco',
       'CR_Projection/Manual_Tools', 'CR_Projection_of_fragments',
       'CR_Suspended_Loads', 'CR_Traffic', 'CR_Vehicles_and_Mobile_Equipment',
       'CR_Venomous_Animals', 'CR_remains_of_choco', 'TFIDF_activity', 'TFIDF_area',
       'TFIDF_causing', 'TFIDF_employee', 'TFIDF_hand', 'TFIDF_injury',
       'TFIDF_left', 'TFIDF_operator', 'TFIDF_right', 'TFIDF_time',
       'TFIDF_causing injury', 'TFIDF_described injury',
       'TFIDF_employee reports', 'TFIDF_finger left', 'TFIDF_injury described',
       'TFIDF_left foot', 'TFIDF_left hand', 'TFIDF_medical center',
       'TFIDF_right hand', 'TFIDF_time accident',
       'TFIDF_causing injury described', 'TFIDF_described time accident',
       'TFIDF_finger left hand', 'TFIDF_finger right hand',
       'TFIDF_generating described injury', 'TFIDF_hand causing injury',
       'TFIDF_injury time accident', 'TFIDF_left hand causing',
       'TFIDF_right hand causing', 'TFIDF_time accident employee', 'Accident Level']

In [ ]:
# Separate input features and target
X_train_smote = df_smote.iloc[:,:-1] # Considering all Predictors
y_train_smote = df_smote.iloc[:,-1:]

In [ ]:
X_train_smote.head(1)

In [ ]:
# Display new accident level counts
y_train_smote['Accident Level'].value_counts()

In [ ]:
# convert integers to dummy variables (i.e. one hot encoded)
y_train_smote_dummy = np_utils.to_categorical(y_train_smote['Accident Level'])
y_train_smote_dummy

#### Varible Tansformation (Normalization and Scaling)

In [ ]:
# Transform independent features
scaler_X = StandardScaler()#StandardScaler()
pipeline = Pipeline(steps=[('s', scaler_X)])
X_train.iloc[:,:6] = pipeline.fit_transform(X_train.iloc[:,:6]) # Scaling only first 6 feautres

X_test.iloc[:,:6] = pipeline.fit_transform(X_test.iloc[:,:6]) # Scaling only first 6 feautres

In [ ]:
X_train.head(3)

#### Use PCA - Extract Principal Components that capture about 95% of the variance in the data

In [ ]:
# generating the covariance matrix and the eigen values for the PCA analysis
cov_matrix = np.cov(X_train.T) # the relevanat covariance matrix
print('Covariance Matrix \n%s', cov_matrix)

#generating the eigen values and the eigen vectors
e_vals, e_vecs = np.linalg.eig(cov_matrix)
print('Eigenvectors \n%s' %e_vecs)
print('\nEigenvalues \n%s' %e_vals)

In [ ]:
# the "cumulative variance explained" analysis 
tot = sum(e_vals)
var_exp = [( i /tot ) * 100 for i in sorted(e_vals, reverse=True)]
cum_var_exp = np.cumsum(var_exp)
print("Cumulative Variance Explained", cum_var_exp)

In [ ]:
# Plotting the variance expalained by the principal components and the cumulative variance explained.
plt.figure(figsize=(20 , 5))
plt.bar(range(1, e_vals.size + 1), var_exp, alpha = 0.5, align = 'center', label = 'Individual explained variance')
plt.step(range(1, e_vals.size + 1), cum_var_exp, where='mid', label = 'Cumulative explained variance')
plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Components')
plt.legend(loc = 'best')
plt.tight_layout()
plt.show()

In [ ]:
# Capturing 90% variance of the data
pca = PCA(n_components = 0.90)
X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)

In [ ]:
print(X_train_reduced.shape)
print(X_test_reduced.shape)

[Table of Contents](#table-of-contents)

<a id="ml-models"></a>
## 10. Design, train and test machine learning classifiers

#### Here we can use the DummyClassifier to predict all accident levels just to show how misleading accuracy can be.

In [ ]:
# DummyClassifier to predict all Accident levels
dummy = DummyClassifier(strategy='stratified').fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

# checking unique labels
print('Unique predicted labels: ', (np.unique(dummy_pred)))

# checking accuracy
print('Test score: ', accuracy_score(y_test, dummy_pred))

In [ ]:
# Checking unique values
predictions = pd.DataFrame(dummy_pred)
predictions[0].value_counts()

#### Define MultiClass-Logloss

In [ ]:
def multiclass_logloss(actual, predicted, eps=1e-15):
    """Multi class version of Logarithmic Loss metric.
    :param actual: Array containing the actual target classes
    :param predicted: Matrix with class predictions, one probability per class
    """
    # Convert 'actual' to a binary array if it's not already:
    if len(actual.shape) == 1:
        actual2 = np.zeros((actual.shape[0], predicted.shape[1]))
        for i, val in enumerate(actual):
            actual2[i, val] = 1
        actual = actual2

    clip = np.clip(predicted, eps, 1 - eps)
    rows = actual.shape[0]
    vsota = np.sum(actual * np.log(clip))
    return -1.0 / rows * vsota

#### Train and test model

In [ ]:
def train_test_model(model, method, X_train, X_test, y_train, y_test, of_type, index, scale, report, save_model):
    
    if report == "yes":
        print (model)
        print ("***************************************************************************")

    if method == 'CatBoostClassifier' or method == 'LGBMClassifier':

      model.fit(X_train, y_train) # Fit the model on Training set
    else:
      model.fit(X_train, y_train) # Fit the model on Training set

    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score, precision_score
    
    if of_type == "coef":
        # Intercept and Coefficients
        print("The intercept for our model is {}".format(model.intercept_), "\n")
        
        for idx, col_name in enumerate(X_train.columns):
            print("The coefficient for {} is {}".format(col_name, model.coef_.ravel()[idx]))

    y_pred = model.predict(X_test) # Predict on Test set

    # Initialise mc_logloss
    mc_logloss = 1.00
    if method != 'RidgeClassifier':
      y_predictions = model.predict_proba(X_test)

    train_accuracy_score = model.score(X_train, y_train)
    test_accuracy_score = model.score(X_test, y_test)

    precision_score = precision_score(y_test, y_pred, average='weighted')
    recall_score = recall_score(y_test, y_pred, average='weighted')
    f1_score = f1_score(y_test, y_pred, average='weighted')

    if method != 'RidgeClassifier':
      mc_logloss = multiclass_logloss(y_test, y_predictions, eps=1e-15)

    if report == "yes":
      # Model - Confusion matrix
      model_cm = confusion_matrix(y_test, y_pred)

      sns.heatmap(model_cm, annot=True,  fmt='.2f', xticklabels = ["I", "II", "III", "IV", "V"] , yticklabels = ["I", "II", "III", "IV", "V"] )
      plt.ylabel('Actual')
      plt.xlabel('Predicted')
      plt.show()

      # Model - Classification report
      model_cr = classification_report(y_test, y_pred)
      print(model_cr)

    # Store the accuracy results for each model in a dataframe for final comparison
    resultsDf = pd.DataFrame({'Method': method, 'Train Accuracy': train_accuracy_score, 'Test Accuracy': test_accuracy_score, 
                              'Precision': precision_score, 'Recall': recall_score, 'F1-Score': f1_score, 
                              'Multi-Class Logloss': mc_logloss}, index=[index])
    
    # Save the model
    if save_model == "yes":
      filename = 'finalised_model.sav'
      pickle.dump(model, open(filename, 'wb'))
      
    return resultsDf  # return all the metrics along with predictions

#### Train and test all models

In [ ]:
import lightgbm as lgb

def train_test_allmodels(X_train_common, X_test_common, y_train, y_test, scale):

    # define classification models
    models=[['LogisticRegression',LogisticRegression(solver='lbfgs', multi_class='multinomial', random_state = 1)],
        ['RidgeClassifier',RidgeClassifier(random_state = 1)],
        #['Lasso',Lasso(random_state = 1)],
        ['KNeighborsClassifier',KNeighborsClassifier(n_neighbors = 3)],
        ['SVC',SVC(kernel = 'rbf', probability=True)],
        ['DecisionTreeClassifier',DecisionTreeClassifier(criterion = 'gini', random_state=1)],
        ['RandomForestClassifier',RandomForestClassifier(n_estimators=10, random_state=1)],
        ['BaggingClassifier',BaggingClassifier(n_estimators=30, max_samples=0.75, random_state=1, oob_score=True)],
        ['ExtraTreesClassifier',ExtraTreesClassifier(n_estimators = 50, criterion='entropy', max_features='auto', min_samples_split=2, 
                                 bootstrap=True, oob_score=True)],
        ['AdaBoostClassifier',AdaBoostClassifier(n_estimators=100, learning_rate=0.25, random_state=1)],
        ['GradientBoostingClassifier',GradientBoostingClassifier(loss='deviance', n_estimators=50, learning_rate=0.1, validation_fraction=0.2, 
                                       random_state=1)],
        ['CatBoostClassifier',CatBoostClassifier(task_type= 'GPU', loss_function="MultiClass", random_state=1, verbose=0)],
                                                #early_stopping_rounds = 30)],
        ['LGBMClassifier',LGBMClassifier(random_state=1, metric = "multi_logloss", objective="multiclass")],
                                         #early_stopping_rounds = 30)],
        ['XGBClassifier',XGBClassifier(min_child_weight = 7, max_depth = 6, objective="multi:softmax", learning_rate = 0.1, gamma = 0.4, 
                                       colsample_bytree = 0.5)]
    ]

    resultsDf_common = pd.DataFrame()
    i = 1
    for name, classifier in models:
        # Train and Test the model
        reg_resultsDf = train_test_model(classifier, name, X_train_common, X_test_common, y_train, y_test, 'none', i, scale, 'no', 'no')

        # Store the accuracy results for each model in a dataframe for final comparison
        resultsDf_common = pd.concat([resultsDf_common, reg_resultsDf])
        i = i+1

    return resultsDf_common

#### Model with Hyperparameter Tuning

In [ ]:
def hyperparameterstune_model(name, model, X_train, y_train, param_grid):
    
    start = time.time()  # note the start time 
    
    # Before starting with grid search we need to create a scoring function. This is accomplished using the make_scorer function of scikit-learn.
    mll_scorer = metrics.make_scorer(multiclass_logloss, greater_is_better=False, needs_proba=True)

    # define grid search
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    if name == 'LGBMClassifier':
      grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, n_iter=100, n_jobs=-1, cv=cv, 
                                       scoring = mll_scorer, error_score=0)
    else:
      grid_search = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=cv, 
                                 scoring = mll_scorer, error_score=0)
      
    model_grid_result = grid_search.fit(X_train, y_train)

    # summarize results
    print("Best F1_Score: %f using %s" % (model_grid_result.best_score_, model_grid_result.best_params_))
    means = model_grid_result.cv_results_['mean_test_score']
    stds = model_grid_result.cv_results_['std_test_score']
    params = model_grid_result.cv_results_['params']
    for mean, stdev, param in zip(means, stds, params):
      if param == model_grid_result.best_params_:
        print("%f (%f) with: %r" % (mean, stdev, param))
        print("95% Confidence interval range: ({0:.4f} %, {1:.4f} %)".format(mean-(2*stdev), mean+(2*stdev)))

    end = time.time()  # note the end time
    duration = end - start  # calculate the total duration
    print("Total duration" , duration, "\n")
    
    return model_grid_result.best_estimator_

#### 1. Modelling - Logistic Regression

In [ ]:
# For multiclass problems, only 'newton-cg', 'sag', 'saga' and 'lbfgs' handle multinomial loss; 'liblinear' is limited to one-versus-rest schemes.

resultsDf = pd.DataFrame()

# Building a Linear Regression model
lr = LogisticRegression(solver='lbfgs', multi_class='multinomial', random_state = 1)
                                                     
# Train and Test the model
resultsDf = train_test_model(lr, 'Logistic Regression without Sampling', X_train, X_test, y_train, y_test, 'none', 1, 'no', 'yes', 'no')

# Store the accuracy results for each model in a dataframe for final comparison
resultsDf

#### 2. Decision Tree - Random Forest Classifier

* While in every machine learning problem, it’s a good rule of thumb to try a variety of algorithms, it can be especially beneficial with imbalanced datasets. Decision trees frequently perform well on imbalanced data. They work by learning a hierarchy of if/else questions and this can force both classes to be addressed.

In [ ]:
# Building a Random Forest Classifier on Training set
rfc_model = RandomForestClassifier(n_estimators=10, random_state=1)

# Train and Test the model
rf_df = train_test_model(rfc_model, 'Random Forest with original data', X_train, X_test, y_train, y_test, 'none', 2, 'no', 'yes', 'no')

#Store the accuracy results for each model in a dataframe for final comparison
resultsDf = pd.concat([resultsDf,rf_df])
resultsDf

#### 3. Modelling - Logistic Regression - Oversampling

In [ ]:
# Building a Linear Regression model
lr = LogisticRegression(solver='lbfgs', multi_class='multinomial', random_state = 1)
                                                     
# Train and Test the model
lr_df = train_test_model(lr, 'Logistic Regression with Sampling', X_train_up, X_test, y_train_up, y_test, 'none', 3, 'no', 'yes', 'no')

#Store the accuracy results for each model in a dataframe for final comparison
resultsDf = pd.concat([resultsDf,lr_df])
resultsDf

#### 4. Modelling - Logistic Regression - SMOTE

In [ ]:
# Building a Linear Regression model
lr = LogisticRegression(solver='lbfgs', multi_class='multinomial', random_state = 1)
                                                     
# Train and Test the model
lr_smote_df = train_test_model(lr, 'Logistic Regression with SMOTE', X_train_smote, X_test, y_train_smote, y_test, 'none', 4, 'no', 'yes', 'no')

#Store the accuracy results for each model in a dataframe for final comparison
resultsDf = pd.concat([resultsDf,lr_smote_df])
resultsDf

[Table of Contents](#table-of-contents)

<a id="ml-models-original-data"></a>
#### All models - Original data

In [ ]:
# Train and Test all models with Lasso interaction terms
train_test_allmodels(X_train, X_test, y_train, y_test, 'no')

By comparing the results from all above methods, we can select the best method as AdaBoost classifier with f1-score 65.38%

<a id="ml-models-oversampling-data"></a>
#### All models - Oversampling data

In [ ]:
# Train and Test all models with Lasso interaction terms
train_test_allmodels(X_train_up, X_test, y_train_up, y_test, 'no')

By comparing the results from all above methods, we can select best method as Ridge classifier with f1-score 62.68% and all other methods are over fitting the training data.

<a id="ml-models-smote-data"></a>
#### All models - SMOTE data

In [ ]:
# Train and Test all models with Lasso interaction terms
train_test_allmodels(X_train_smote, X_test, y_train_smote, y_test, 'no')

By comparing the results from all above methods, all are over fitting the training data.

<a id="ml-models-hyperparam-tuning"></a>
#### Hyperparameter tuning with original features

In [ ]:
# define regressor models
models=[['LogisticRegression',LogisticRegression()],
    ['Ridge',RidgeClassifier()],
    ['KNeighborsClassifier',KNeighborsClassifier()],
    ['SVC',SVC()]
    ['RandomForestClassifier',RandomForestClassifier()],
    ['BaggingClassifier',BaggingClassifier()],
    ['ExtraTreesClassifier',ExtraTreesClassifier()],
    ['AdaBoostClassifier',AdaBoostClassifier()],
    ['GradientBoostingClassifier',GradientBoostingClassifier()],
    ['CatBoostClassifier',CatBoostClassifier(verbose=False)],
    ['LGBMClassifier',LGBMClassifier(verbose=False)],
    ['XGBClassifier',XGBClassifier()]
]

# define model parameters

lr_param_grid = {'solver': ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
                 'penalty': ['l2'],
                 #'penalty': ['none', 'l1', 'l2', 'elasticnet'],
                 'C': [100, 10, 1.0, 0.1, 0.01]}
                 #'class_weight':['none','balanced'],
                 #'multi_class':['ovr', 'multinomial']}
ridge_param_grid = {'alpha': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                    'class_weight':['none','balanced'],
                    'solver': ['svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']}
#lasso_param_grid = {'alpha': [0.02, 0.024, 0.025, 0.026, 0.03]}
knn_param_grid = {'n_neighbors': range(3, 21, 2),
                 'weights': ['uniform', 'distance'],
                 'metric': ['euclidean', 'manhattan', 'minkowski']}
svc_param_grid = {'kernel': ['poly', 'rbf', 'sigmoid'],
                 'C': [50, 10, 1.0, 0.1, 0.01],
                 'gamma': ['scale'],
                 'decision_function_shape': ['ovo', 'ovr']}
rf_param_grid = {'n_estimators': [10, 100, 1000],
                 'max_features': ['auto', 'sqrt', 'log2']}              
                 #'class_weight':['balanced','balanced_subsample','none']}
bag_param_grid = {'n_estimators': [10, 100, 1000],
                 'max_samples': np.arange(0.7, 0.8, 0.05)}
et_param_grid = {'n_estimators': np.arange(10,100,10),
                 'max_features': ['auto', 'sqrt', 'log2'],
                 'min_samples_split': np.arange(2,15,1)}
                 #'class_weight':['balanced','balanced_subsample','none']}
adb_param_grid = {'n_estimators': np.arange(30,100,10),
                 'learning_rate': np.arange(0.1,1,0.5)}
gb_param_grid = {'n_estimators': [10, 50, 100, 500],
                 'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1.0],
                 'subsample':[0.5, 0.7, 1.0],
                 'max_depth': [3, 7, 9]}
catb_param_grid = {'task_type': 'GPU','depth': [4, 7, 10],
                  'learning_rate' : [0.03, 0.1, 0.15],
                  'l2_leaf_reg': [1,4,9],
                  'early_stopping_rounds':[50],
                  'iterations': [300],
                  'loss_function':['MultiClass']}
lightgbm_param_grid = {'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1.0],
                  'n_estimators': [10, 50, 100, 500, 1000, 5000],
                  'min_child_samples': sp_randint(100, 500), 
                  'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3, 1e4],
                  'boosting_type':['gbdt', 'dart', 'goss'],
                  'bagging_fraction': (0.5, 1),
                  'bagging_frequency' : (5, 8),
                  'feature_fraction': (0.5, 0.8),
                  'max_depth': (10, 13),
                  'min_data_in_leaf': (90, 120),
                  'num_leaves':(1200, 1550),
                  'metric': ['multi_logloss'],
                  'objective': ['multiclass'],
                  'num_class': [5],
                  'early_stopping_rounds':[50],
                  'verbosity':[1]}
xgb_param_grid = {'learning_rate': [0.05, 0.10, 0.15, 0.20, 0.25, 0.30 ],
                  'max_depth' : [3, 4, 5, 6, 8, 10, 12, 15],
                  'min_child_weight': [ 1, 3, 5, 7],
                  'gamma': [0.0, 0.1, 0.2 , 0.3, 0.4],
                  'colsample_bytree': [ 0.3, 0.4, 0.5 , 0.7],
                  'objective': ['multi:softmax'],
                  'eval_metric': ['mlogloss'],
                  'num_class': [5]}

for name, classifier in models:
    if name == 'LogisticRegression':
        lr_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, lr_param_grid)
    elif name == 'Ridge':
        ridge_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, lasso_param_grid)
    elif name == 'KNeighborsClassifier':
        knn_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, knn_param_grid)
    elif name == 'SVC':
        svc_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, svc_param_grid)
    elif name == 'RandomForestClassifier':
        rf_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, rf_param_grid)
    elif name == 'BaggingClassifier':
        bag_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, bag_param_grid)
    elif name == 'ExtraTreesClassifier':
        et_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, et_param_grid)
    elif name == 'AdaBoostClassifier':
        adb_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, adb_param_grid)
    elif name == 'GradientBoostingClassifier':
        gb_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, gb_param_grid)
    elif name == 'CatBoostClassifier':
        catb_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, catb_param_grid)
    elif name == 'LGBMClassifier':
        lightgbm_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, lightgbm_param_grid)
    elif name == 'XGBClassifier':
        xgb_best_estimator = hyperparameterstune_model(name, classifier, X_train, y_train, xgb_param_grid)

1. Logistic Regression

  Best F1_Score: 0.629247 using {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
  0.629247 (0.026945) with: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
  95% Confidence interval range: (0.5754 %, 0.6831 %)
  Total duration 428.40893173217773 

2. Ridge

  Best F1_Score: 0.616087 using {'alpha': 0.02}
  0.616087 (0.023211) with: {'alpha': 0.02}
  95% Confidence interval range: (0.5697 %, 0.6625 %)
  Total duration 2.5354909896850586 

3. KNeighborsClassifier

  Best F1_Score: 0.629247 using {'metric': 'euclidean', 'n_neighbors': 13, 'weights': 'uniform'}
  0.629247 (0.026945) with: {'metric': 'euclidean', 'n_neighbors': 13, 'weights': 'uniform'}
  95% Confidence interval range: (0.5754 %, 0.6831 %)
  Total duration 12.201840877532959 

4. SVC

  Best F1_Score: 0.629247 using {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
  0.629247 (0.026945) with: {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
  95% Confidence interval range: (0.5754 %, 0.6831 %)
  Total duration 7.190079927444458 

5. RandomForestClassifier

  Best F1_Score: 0.622235 using {'max_features': 'sqrt', 'n_estimators': 1000}
  0.622235 (0.028932) with: {'max_features': 'sqrt', 'n_estimators': 1000}
  95% Confidence interval range: (0.5644 %, 0.6801 %)
  Total duration 176.1380627155304 

6. BaggingClassifier

  Best F1_Score: 0.620604 using {'max_samples': 0.75, 'n_estimators': 10}
  0.620604 (0.034210) with: {'max_samples': 0.75, 'n_estimators': 10}
  95% Confidence interval range: (0.5522 %, 0.6890 %)
  Total duration 305.1015794277191 

7. ExtraTreesClassifier

  Best F1_Score: 0.628276 using {'max_features': 'log2', 'min_samples_split': 13, 'n_estimators': 90}
  0.628276 (0.028086) with: {'max_features': 'log2', 'min_samples_split': 13, 'n_estimators': 90}
  95% Confidence interval range: (0.5721 %, 0.6844 %)
  Total duration 688.0113415718079 

8. AdaBoostClassifier

  Best F1_Score: 0.630245 using {'learning_rate': 0.1, 'n_estimators': 70}
  0.630245 (0.030167) with: {'learning_rate': 0.1, 'n_estimators': 70}
  95% Confidence interval range: (0.5699 %, 0.6906 %)
  Total duration 51.544673681259155 

9. GradientBoostingClassifier

  Best F1_Score: 0.613227 using {'learning_rate': 0.1, 'n_estimators': 30}
  0.613227 (0.033099) with: {'learning_rate': 0.1, 'n_estimators': 30}
  95% Confidence interval range: (0.5470 %, 0.6794 %)
  Total duration 165.68569326400757 

10. CatBoostClassifier

  Best F1_Score: 0.629247 using {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 4, 'learning_rate': 0.03}
  0.629247 (0.026945) with: {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 4, 'learning_rate': 0.03}
  95% Confidence interval range: (0.5754 %, 0.6831 %)
  Total duration 6367.161168336868 

11. LGBMClassifier

  Best F1_Score: 0.629247 using {'bagging_fraction': 0.5, 'bagging_frequency': 8, 'boosting_type': 'dart', 'early_stopping_rounds': 200, 'feature_fraction': 0.8, 'learning_rate': 0.0001, 'max_depth': 10, 'metric': 'multi_logloss', 'min_child_samples': 486, 'min_child_weight': 0.01, 'min_data_in_leaf': 90, 'n_estimators': 1000, 'num_class': 5, 'num_leaves': 1550, 'objective': 'multiclass', 'verbosity': 1}
  0.629247 (0.026945) with: {'bagging_fraction': 0.5, 'bagging_frequency': 8, 'boosting_type': 'dart', 'early_stopping_rounds': 200, 'feature_fraction': 0.8, 'learning_rate': 0.0001, 'max_depth': 10, 'metric': 'multi_logloss', 'min_child_samples': 486, 'min_child_weight': 0.01, 'min_data_in_leaf': 90, 'n_estimators': 1000, 'num_class': 5, 'num_leaves': 1550, 'objective': 'multiclass', 'verbosity': 1}
  95% Confidence interval range: (0.5754 %, 0.6831 %)
  Total duration 2430.5182700157166 

[Table of Contents](#table-of-contents)

<a id="ml-models-bootstrap-sampling"></a>
#### Bootstrap Sampling - RandomForestClassifier

In [ ]:
ind_feat_df.drop(['Accident Level','Potential Accident Level'], axis = 1, inplace=True) # Considering all Predictors

# Consider only top 30 GLOVE features
ind_feat_df = ind_feat_df.join(y.reset_index(drop=True))
ind_feat_df.head(2)

In [ ]:
values = ind_feat_df.values

# Number of bootstrap samples to create
n_iterations = 1000        

# size of a bootstrap sample
n_size = int(len(ind_feat_df) * 1)    

# run bootstrap
# empty list that will hold the scores for each bootstrap iteration
rf_stats = list()   
for i in range(n_iterations):
    # prepare train and test sets
    train = resample(values, n_samples=n_size)  # Sampling with replacement 
    test = np.array([x for x in values if x.tolist() not in train.tolist()])  # picking rest of the data not considered in sample
    
     # fit model
    rfTree = RandomForestClassifier(n_estimators=100)

    # fit against independent variables and corresponding target values
    rfTree.fit(train[:,:-1], train[:,-1]) 

    # Take the target column for all rows in test set
    y_bs_test = test[:,-1]  

    # evaluate model
    # predict based on independent variables in the test data
    score = rfTree.score(test[:, :-1] , y_bs_test)
    predictions = rfTree.predict(test[:, :-1]) 

    rf_stats.append(score)

In [ ]:
# plot scores
plt.hist(rf_stats)
plt.show()

# confidence intervals
alpha = 0.95                             # for 95% confidence 
p = ((1.0-alpha)/2.0) * 100              # tail regions on right and left .25 on each side indicated by P value (border)
lower = max(0.0, np.percentile(rf_stats, p))  
p = (alpha+((1.0-alpha)/2.0)) * 100
upper = min(1.0, np.percentile(rf_stats, p))
print('%.1f confidence interval %.1f%% and %.1f%%' % (alpha*100, lower*100, upper*100))

#### Bootstrap Sampling - AdaBoostClassifier

In [ ]:
values = ind_feat_df.values

# Number of bootstrap samples to create
n_iterations = 1000        

# size of a bootstrap sample
n_size = int(len(ind_feat_df) * 1)    

# run bootstrap
# empty list that will hold the scores for each bootstrap iteration
adab_stats = list()   
for i in range(n_iterations):
    # prepare train and test sets
    train = resample(values, n_samples=n_size)  # Sampling with replacement 
    test = np.array([x for x in values if x.tolist() not in train.tolist()])  # picking rest of the data not considered in sample
    
     # fit model
    adabTree = AdaBoostClassifier(n_estimators=100, learning_rate=0.25, random_state=1)

    # fit against independent variables and corresponding target values
    adabTree.fit(train[:,:-1], train[:,-1]) 

    # Take the target column for all rows in test set
    y_bs_test = test[:,-1]  

    # evaluate model
    # predict based on independent variables in the test data
    score = adabTree.score(test[:, :-1] , y_bs_test)
    predictions = adabTree.predict(test[:, :-1]) 

    adab_stats.append(score)

In [ ]:
# plot scores
plt.hist(adab_stats)
plt.show()

# confidence intervals
alpha = 0.95                             # for 95% confidence 
p = ((1.0-alpha)/2.0) * 100              # tail regions on right and left .25 on each side indicated by P value (border)
lower = max(0.0, np.percentile(adab_stats, p))  
p = (alpha+((1.0-alpha)/2.0)) * 100
upper = min(1.0, np.percentile(adab_stats, p))
print('%.1f confidence interval %.1f%% and %.1f%%' % (alpha*100, lower*100, upper*100))

#### Bootstrap Sampling - XGBClassifier

In [ ]:
values = ind_feat_df.values

# Number of bootstrap samples to create
n_iterations = 1000        

# size of a bootstrap sample
n_size = int(len(ind_feat_df) * 1)    

# run bootstrap
# empty list that will hold the scores for each bootstrap iteration
xgb_stats = list()   
for i in range(n_iterations):
    # prepare train and test sets
    train = resample(values, n_samples=n_size)  # Sampling with replacement 
    test = np.array([x for x in values if x.tolist() not in train.tolist()])  # picking rest of the data not considered in sample
    
     # fit model
    xgbTree = XGBClassifier(max_depth = 6, objective="multi:softmax", learning_rate = 0.1, gamma = 0.4)

    # fit against independent variables and corresponding target values
    xgbTree.fit(train[:,:-1], train[:,-1]) 

    # Take the target column for all rows in test set
    y_bs_test = test[:,-1]  

    # evaluate model
    # predict based on independent variables in the test data
    score = xgbTree.score(test[:, :-1] , y_bs_test)
    predictions = xgbTree.predict(test[:, :-1]) 

    xgb_stats.append(score)

In [ ]:
# plot scores
plt.hist(xgb_stats)
plt.show()

# confidence intervals
alpha = 0.95                             # for 95% confidence 
p = ((1.0-alpha)/2.0) * 100              # tail regions on right and left .25 on each side indicated by P value (border)
lower = max(0.0, np.percentile(xgb_stats, p))  
p = (alpha+((1.0-alpha)/2.0)) * 100
upper = min(1.0, np.percentile(xgb_stats, p))
print('%.1f confidence interval %.1f%% and %.1f%%' % (alpha*100, lower*100, upper*100))

[Table of Contents](#table-of-contents)

<a id="ann-models"></a>
## 11. Design, train and test Neural networks classifiers

In [ ]:
# disable keras warnings
tf.get_logger().setLevel('ERROR')

#### Get ANN Multiclass Classification Metrics

In [ ]:
# get the accuracy, precision, recall, f1 score from model
def get_classification_metrics(model, X_test, y_test, target_type):
  
  # predict probabilities for test set
  yhat_probs = model.predict(X_test, verbose=0) # Multiclass

  # predict crisp classes for test set
  if target_type == 'multi_class':
    yhat_classes = model.predict_classes(X_test, verbose=0) # Multiclass
  else:
    yhat_classes = (np.asarray(model.predict(X_test))).round() # Multilabel

  # reduce to 1d array
  yhat_probs = yhat_probs[:, 0]

  # accuracy: (tp + tn) / (p + n)
  accuracy = accuracy_score(y_test, yhat_classes)

  # precision tp / (tp + fp)
  precision = precision_score(y_test, yhat_classes, average='micro')

  # recall: tp / (tp + fn)
  recall = recall_score(y_test, yhat_classes, average='micro')

  # f1: 2 tp / (2 tp + fp + fn)
  f1 = f1_score(y_test, yhat_classes, average='micro')

  return accuracy, precision, recall, f1

In [ ]:
class Metrics(tf.keras.callbacks.Callback):

    def __init__(self, validation_data=()):
        super().__init__()
        self.validation_data = validation_data

    def on_train_begin(self, logs={}):
        self.val_f1s = []
        self.val_recalls = []
        self.val_precisions = []

    def on_epoch_end(self, epoch, logs={}):
        xVal, yVal, target_type = self.validation_data
        if target_type == 'multi_class':
          val_predict_classes = model.predict_classes(xVal, verbose=0) # Multiclass
        else:
          val_predict_classes = (np.asarray(self.model.predict(xVal))).round() # Multilabel
        
        
        val_targ = yVal

        _val_f1 = f1_score(val_targ, val_predict_classes, average='micro')
        _val_recall = recall_score(val_targ, val_predict_classes, average='micro')
        _val_precision = precision_score(val_targ, val_predict_classes, average='micro')
        self.val_f1s.append(_val_f1)
        self.val_recalls.append(_val_recall)
        self.val_precisions.append(_val_precision)
        #print("— train_f1: %f — train_precision: %f — train_recall %f" % (_val_f1, _val_precision, _val_recall))
        return

<a id="ann-models-clas-to-num-problem"></a>
#### Convert Classification to Numeric problem

In this section, we will create a classification model that uses categorical columns and tf-idf features from accident description and label encoded target variable. We can use simple densely connected neural networks to make predictions.

Since we have ordinal relationship between each category in target variable, I have considered this one as numerical/regression problem and try to observe the ANN behaviour.


In [ ]:
# fix random seed for reproducibility
seed = 7
np.random.seed(seed)
tf.random.set_seed(seed)

# define the model
model = Sequential()
model.add(Dense(50, input_dim=X_train.shape[1], activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(100, activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(150, activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(40, activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(1, activation='linear'))

# compile the keras model
#opt = optimizers.Adam(lr=1e-3)
opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='mse', optimizer=opt, metrics=['accuracy'])

# Use earlystopping
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, min_delta=0.001)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

# fit the keras model on the dataset
training_history = model.fit(X_train, y_train, epochs=100, batch_size=8, verbose=1, validation_data=(X_test, y_test), callbacks=[rlrp])

In [ ]:
model.summary()

In [ ]:
# evaluate the keras model
_, accuracy = model.evaluate(X_test, y_test, batch_size=8, verbose=0)
print('Test accuracy: %.2f' % (accuracy*100))

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot  (epochs, training_history.history['loss'], label = 'train')
plt.plot  (epochs, training_history.history['val_loss'], label = 'val')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is underfit model, it can be identified from the learning curve of the training loss only. It is showing noisy values of relatively high loss, indicating that the model was unable to learn the training dataset at all and model does not have a suitable capacity for the complexity of the dataset.

<a id="ann-models-multi-class"></a>
#### Multiclass classification - Target variable - One hot encoded

In this section, we will create a classification model that uses categorical columns and tf-idf features from accident description and one-hot encoded target variable. We can use simple densely connected neural networks to make predictions.

In [ ]:
# fix random seed for reproducibility
reset_random_seeds()
#param = 1e-9
param = 1e-4

# define the model
model = Sequential()

model.add(Dense(10, input_dim=X_train.shape[1], activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param),
                kernel_constraint=unit_norm()))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(Dense(10, activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param), 
                kernel_constraint=unit_norm()))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(5, activation='softmax', kernel_regularizer=l2(param), 
                kernel_constraint=unit_norm())) # Multilabel

# compile the keras model
#opt = optimizers.Adamax(lr=0.01)
opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['categorical_accuracy'])

# Use earlystopping
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=7, min_delta=1E-3)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

target_type = 'multi_label'
metrics = Metrics(validation_data=(X_train, y_train_dummy, target_type))

# fit the keras model on the dataset
training_history = model.fit(X_train, y_train_dummy, epochs=100, batch_size=8, verbose=1, validation_data=(X_test, y_test_dummy), callbacks=[rlrp, metrics])

In [ ]:
model.summary()

In [ ]:
# evaluate the keras model
_, train_accuracy = model.evaluate(X_train, y_train_dummy, batch_size=8, verbose=0)
_, test_accuracy = model.evaluate(X_test, y_test_dummy, batch_size=8, verbose=0)

print('Train accuracy: %.2f' % (train_accuracy*100))
print('Test accuracy: %.2f' % (test_accuracy*100))

In [ ]:
accuracy, precision, recall, f1 = get_classification_metrics(model, X_test, y_test_dummy, target_type)
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot(epochs, training_history.history['loss'], label = 'train')
plt.plot(epochs, training_history.history['val_loss'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is good fit, it is identified by a training and validation loss that decreases to a point of stability with a minimal gap between the two final loss values. The loss of the model will almost always be lower on the training dataset than the validation dataset.

In [ ]:
# plot accuracy learning curves
plt.plot(epochs, training_history.history['categorical_accuracy'], label = 'train')
plt.plot(epochs, training_history.history['val_categorical_accuracy'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation accuracy')

We could see it accuracy continually rise during training. As expected, we see the learning curves for accuracy on the test dataset plateau, indicating that the model has no longer overfit the training dataset and it is generalized model.

In [ ]:
# serialize model to JSON
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
    
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model weights to disk")

# Save the model in h5 format 
model.save("finalized_keras_model.h5")
print("Saved model to disk")

#### Multiclass classification - Target variable - One hot encoded with SMOTE data

In this section, we will create a classification model that uses categorical columns and tf-idf features from accident description and one-hot encoded target variable. We can use simple densely connected neural networks to make predictions.

In [ ]:
# fix random seed for reproducibility
reset_random_seeds()
#param = 1e-9
param = 1e-4

# define the model
model = Sequential()

model.add(Dense(10, input_dim=X_train_smote.shape[1], activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param),
                kernel_constraint=unit_norm()))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(Dense(10, activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param), 
                kernel_constraint=unit_norm()))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(5, activation='softmax', kernel_regularizer=l2(param), 
                kernel_constraint=unit_norm())) # Multilabel

# compile the keras model
#opt = optimizers.Adamax(lr=0.01)
opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['categorical_accuracy'])

# Use earlystopping
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=7, min_delta=1E-3)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

target_type = 'multi_label'
metrics = Metrics(validation_data=(X_train_smote, y_train_smote_dummy, target_type))

# fit the keras model on the dataset
training_history = model.fit(X_train_smote, y_train_smote_dummy, epochs=100, batch_size=8, verbose=1, validation_data=(X_test, y_test_dummy), callbacks=[rlrp, metrics])

In [ ]:
model.summary()

In [ ]:
# evaluate the keras model
_, train_accuracy = model.evaluate(X_train_smote, y_train_smote_dummy, batch_size=8, verbose=0)
_, test_accuracy = model.evaluate(X_test, y_test_dummy, batch_size=8, verbose=0)

print('Train accuracy: %.2f' % (train_accuracy*100))
print('Test accuracy: %.2f' % (test_accuracy*100))

In [ ]:
accuracy, precision, recall, f1 = get_classification_metrics(model, X_test, y_test_dummy, target_type)
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot(epochs, training_history.history['loss'], label = 'train')
plt.plot(epochs, training_history.history['val_loss'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is overfit model, it can be identified from the learning curve of the training and validation loss only.

In [ ]:
# plot accuracy learning curves
plt.plot(epochs, training_history.history['categorical_accuracy'], label = 'train')
plt.plot(epochs, training_history.history['val_categorical_accuracy'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation accuracy')

In [ ]:
# serialize model to JSON
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
    
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model weights to disk")

# Save the model in h5 format 
model.save("finalized_keras_model.h5")
print("Saved model to disk")

[Table of Contents](#table-of-contents)

<a id="nlp-models"></a>
## 12. Design, train and test RNN or LSTM classifiers

#### Architecture

1. Create a model with Text inputs only.
2. Create a model with Categorical inputs only.
3. Create a model with Multiple inputs.

<a id="nlp-models-text-input"></a>
##### 1. Creating a Model with Text Inputs Only

In this section, we will create a classification model that uses accident description column alone.

In [ ]:
# Select input and output features
X_text = industry_df['Cleaned_Description']
y_text = industry_df['Accident Level']

In [ ]:
# Encode labels in column 'Accident Level'.
y_text = LabelEncoder().fit_transform(y_text)

In [ ]:
# Divide our data into testing and training sets:
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(X_text, y_text, test_size = 0.20, random_state = 1, stratify = y_text)

print('X_text_train shape : ({0})'.format(X_text_train.shape[0]))
print('y_text_train shape : ({0},)'.format(y_text_train.shape[0]))
print('X_text_test shape : ({0})'.format(X_text_test.shape[0]))
print('y_text_test shape : ({0},)'.format(y_text_test.shape[0]))

In [ ]:
# Convert both the training and test labels into one-hot encoded vectors:
y_text_train = np_utils.to_categorical(y_text_train)
y_text_test = np_utils.to_categorical(y_text_test)

In [ ]:
# The first step in word embeddings is to convert the words into thier corresponding numeric indexes.
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_text_train)

X_text_train = tokenizer.texts_to_sequences(X_text_train)
X_text_test = tokenizer.texts_to_sequences(X_text_test)

In [ ]:
# Sentences can have different lengths, and therefore the sequences returned by the Tokenizer class also consist of variable lengths.
# We need to pad the our sequences using the max length.
vocab_size = len(tokenizer.word_index) + 1
print("vocab_size:", vocab_size)

maxlen = 100

X_text_train = pad_sequences(X_text_train, padding='post', maxlen=maxlen)
X_text_test = pad_sequences(X_text_test, padding='post', maxlen=maxlen)

In [ ]:
# We need to load the built-in GloVe word embeddings
embedding_size = 200
embeddings_dictionary = dict()

glove_file = open('../input/glove6b200d/glove.6B.200d.txt', encoding="utf8")

for line in glove_file:
    records = line.split()
    word = records[0]
    vector_dimensions = np.asarray(records[1:], dtype='float32')
    embeddings_dictionary[word] = vector_dimensions

glove_file.close()

embedding_matrix = np.zeros((vocab_size, embedding_size))

for word, index in tokenizer.word_index.items():
    embedding_vector = embeddings_dictionary.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

len(embeddings_dictionary.values())

In [ ]:
reset_random_seeds()

# Build a LSTM Neural Network
deep_inputs = Input(shape=(maxlen,))
embedding_layer = Embedding(vocab_size, embedding_size, weights=[embedding_matrix], trainable=False)(deep_inputs)

LSTM_Layer_1 = Bidirectional(LSTM(128, return_sequences = True))(embedding_layer)
max_pool_layer_1 = GlobalMaxPool1D()(LSTM_Layer_1)
drop_out_layer_1 = Dropout(0.5, input_shape = (256,))(max_pool_layer_1)
dense_layer_1 = Dense(128, activation = 'relu')(drop_out_layer_1)
drop_out_layer_2 = Dropout(0.5, input_shape = (128,))(dense_layer_1)
dense_layer_2 = Dense(64, activation = 'relu')(drop_out_layer_2)
drop_out_layer_3 = Dropout(0.5, input_shape = (64,))(dense_layer_2)

dense_layer_3 = Dense(32, activation = 'relu')(drop_out_layer_3)
drop_out_layer_4 = Dropout(0.5, input_shape = (32,))(dense_layer_3)

dense_layer_4 = Dense(10, activation = 'relu')(drop_out_layer_4)
drop_out_layer_5 = Dropout(0.5, input_shape = (10,))(dense_layer_4)

dense_layer_5 = Dense(5, activation='softmax')(drop_out_layer_5)
#dense_layer_3 = Dense(5, activation='softmax')(drop_out_layer_3)

# LSTM_Layer_1 = LSTM(128)(embedding_layer)
# dense_layer_1 = Dense(5, activation='softmax')(LSTM_Layer_1)
# model = Model(inputs=deep_inputs, outputs=dense_layer_1)

model = Model(inputs=deep_inputs, outputs=dense_layer_5)
#model = Model(inputs=deep_inputs, outputs=dense_layer_3)

opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['acc'])

In [ ]:
print(model.summary())

In [ ]:
plot_model(model, to_file='model_plot1.png', show_shapes=True, show_dtype=True, show_layer_names=True)

In [ ]:
# Use earlystopping
# callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, min_delta=0.001)
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=7, min_delta=1E-3)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

target_type = 'multi_label'
metrics = Metrics(validation_data=(X_text_train, y_text_train, target_type))

# fit the keras model on the dataset
training_history = model.fit(X_text_train, y_text_train, epochs=100, batch_size=8, verbose=1, validation_data=(X_text_test, y_text_test), callbacks=[rlrp, metrics])

In [ ]:
# evaluate the keras model
_, train_accuracy = model.evaluate(X_text_train, y_text_train, batch_size=8, verbose=0)
_, test_accuracy = model.evaluate(X_text_test, y_text_test, batch_size=8, verbose=0)

print('Train accuracy: %.2f' % (train_accuracy*100))
print('Test accuracy: %.2f' % (test_accuracy*100))

In [ ]:
accuracy, precision, recall, f1 = get_classification_metrics(model, X_text_test, y_text_test, target_type)
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot(epochs, training_history.history['loss'], label = 'train')
plt.plot(epochs, training_history.history['val_loss'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is good fit, it is identified by a training and validation loss that decreases to a point of stability with a minimal gap between the two final loss values. The loss of the model will almost always be lower on the training dataset than the validation dataset.

In [ ]:
# plot accuracy learning curves
plt.plot(epochs, training_history.history['acc'], label = 'train')
plt.plot(epochs, training_history.history['val_acc'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation accuracy')

We could see it accuracy continually rise during training. As expected, we see the learning curves for accuracy on the test dataset plateau, indicating that the model has no longer overfit the training dataset and it is generalized model.

**Note: Surprisingly we observe that same f1-score = 73.89 % with accident description alone.**


In [ ]:
# serialize model to JSON
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
    
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model weights to disk")

# Save the model in h5 format 
model.save("finalized_keras_model.h5")
print("Saved model to disk")

[Table of Contents](#table-of-contents)

<a id="nlp-models-cat-features"></a>
##### 2. Creating a Model with Categorical features Only

In this section, we will create a classification model that uses categorical columns alone. Since the data for these columns is well structured and doesn't contain any sequential or spatial pattern, we can use simple densely connected neural networks to make predictions.

In [ ]:
# Select input and output features
X_cat = ind_featenc_df.drop(['Accident Level','Potential Accident Level'], axis = 1)
y_cat = industry_df['Accident Level']

In [ ]:
# Encode labels in column 'Accident Level'.
y_cat = LabelEncoder().fit_transform(y_cat)

In [ ]:
# Divide our data into testing and training sets:
X_cat_train, X_cat_test, y_cat_train, y_cat_test = train_test_split(X_cat, y_cat, test_size = 0.20, random_state = 1, stratify = y_cat)

print('X_cat_train shape : ({0})'.format(X_cat_train.shape[0]))
print('y_cat_train shape : ({0},)'.format(y_cat_train.shape[0]))
print('X_cat_test shape : ({0})'.format(X_cat_test.shape[0]))
print('y_cat_test shape : ({0},)'.format(y_cat_test.shape[0]))

In [ ]:
# Convert both the training and test labels into one-hot encoded vectors:
y_cat_train = np_utils.to_categorical(y_cat_train)
y_cat_test = np_utils.to_categorical(y_cat_test)

In [ ]:
# Variable transformation using StandardScaler
scaler_X = StandardScaler()#StandardScaler()
X_cat_train.iloc[:,:6] = scaler_X.fit_transform(X_cat_train.iloc[:,:6]) # Scaling only first 6 feautres

X_cat_test.iloc[:,:6] = scaler_X.fit_transform(X_cat_test.iloc[:,:6]) # Scaling only first 6 feautres

In [ ]:
# fix random seed for reproducibility
reset_random_seeds()

#param = 1e-9
param = 1e-4

input2 = Input(shape=(X_cat_train.shape[1],))
dense_layer_1 = Dense(10, input_dim=X_cat_train.shape[1], activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param),
                kernel_constraint=unit_norm())(input2)
drop_out_layer_1 = Dropout(0.2)(dense_layer_1)
batch_norm_layer_1 = BatchNormalization()(drop_out_layer_1)
dense_layer_2 = Dense(10, activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param), 
                kernel_constraint=unit_norm())(batch_norm_layer_1)
drop_out_layer_2 = Dropout(0.5)(dense_layer_2)
batch_norm_layer_2 = BatchNormalization()(drop_out_layer_2)
dense_layer_3 = Dense(5, activation='softmax', kernel_regularizer=l2(param), kernel_constraint=unit_norm())(batch_norm_layer_2)

model = Model(inputs=input2, outputs=dense_layer_3)

# compile the keras model
#opt = optimizers.Adamax(lr=0.01)
opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['acc'])

In [ ]:
print(model.summary())

In [ ]:
plot_model(model, to_file='model_plot1.png', show_shapes=True, show_dtype=True, show_layer_names=True)

In [ ]:
# Use earlystopping
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=7, min_delta=1E-3)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

target_type = 'multi_label'
metrics = Metrics(validation_data=(X_cat_train, y_cat_train, target_type))

# fit the keras model on the dataset
training_history = model.fit(X_cat_train, y_cat_train, epochs=100, batch_size=8, verbose=1, validation_data=(X_cat_test, y_cat_test), callbacks=[rlrp, metrics])

In [ ]:
# evaluate the keras model
_, train_accuracy = model.evaluate(X_cat_train, y_cat_train, batch_size=8, verbose=0)
_, test_accuracy = model.evaluate(X_cat_test, y_cat_test, batch_size=8, verbose=0)

print('Train accuracy: %.2f' % (train_accuracy*100))
print('Test accuracy: %.2f' % (test_accuracy*100))

In [ ]:
accuracy, precision, recall, f1 = get_classification_metrics(model, X_cat_test, y_cat_test, target_type)
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot(epochs, training_history.history['loss'], label = 'train')
plt.plot(epochs, training_history.history['val_loss'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is good fit, it is identified by a training and validation loss that decreases to a point of stability with a minimal gap between the two final loss values. The loss of the model will almost always be lower on the training dataset than the validation dataset.

In [ ]:
# plot accuracy learning curves
plt.plot(epochs, training_history.history['acc'], label = 'train')
plt.plot(epochs, training_history.history['val_acc'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation accuracy')

We could see it accuracy continually rise during training. As expected, we see the learning curves for accuracy on the test dataset plateau, indicating that the model has no longer overfit the training dataset and it is generalized model.

In [ ]:
# serialize model to JSON
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
    
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model weights to disk")

# Save the model in h5 format 
model.save("finalized_keras_model.h5")
print("Saved model to disk")

[Table of Contents](#table-of-contents)

<a id="nlp-models-multiple-input"></a>
##### 3. Creating a Model with Multiple Inputs

The first submodel will accept textual input in the form of accident description. This submodel will consist of an input shape layer, an embedding layer, and bidirectional LSTM layer of 128 neurons followed by max pool layer, drop out and dense layers. The second submodel will accept input in the form of meta information which consists of dense, batch norm and drop out layers.

The output from the dropout layer of the first submodel and the output from the batch norm layer of the second submodel will be concatenated together and will be used as concatenated input to another dense layer with 10 neurons. Finally, the output dense layer will have five neuorns corresponding to each accident level.


In [ ]:
# fix random seed for reproducibility
reset_random_seeds()

input_1 = Input(shape=(maxlen,))
embedding_layer   = Embedding(vocab_size, embedding_size, weights=[embedding_matrix], trainable=False)(input_1)
LSTM_Layer_1      = Bidirectional(LSTM(128, return_sequences = True))(embedding_layer)
max_pool_layer_1  = GlobalMaxPool1D()(LSTM_Layer_1)
drop_out_layer_1  = Dropout(0.5, input_shape = (256,))(max_pool_layer_1)
dense_layer_1     = Dense(128, activation = 'relu')(drop_out_layer_1)
drop_out_layer_2  = Dropout(0.5, input_shape = (128,))(dense_layer_1)
dense_layer_2     = Dense(64, activation = 'relu')(drop_out_layer_2)
drop_out_layer_3  = Dropout(0.5, input_shape = (64,))(dense_layer_2)

dense_layer_3     = Dense(32, activation = 'relu')(drop_out_layer_3)
drop_out_layer_4  = Dropout(0.5, input_shape = (32,))(dense_layer_3)

dense_layer_4     = Dense(10, activation = 'relu')(drop_out_layer_4)
drop_out_layer_5 = Dropout(0.5, input_shape = (10,))(dense_layer_4)

#-------------------------------------------------------------------------------
param = 1e-4

input_2 = Input(shape=(X_cat_train.shape[1],))
dense_layer_5       = Dense(10, input_dim=X_cat_train.shape[1], activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param),
                      kernel_constraint=unit_norm())(input_2)
drop_out_layer_6    = Dropout(0.2)(dense_layer_5)
batch_norm_layer_1  = BatchNormalization()(drop_out_layer_6)
dense_layer_6       = Dense(10, activation='relu', kernel_initializer='he_uniform', kernel_regularizer=l2(param), 
                            kernel_constraint=unit_norm())(batch_norm_layer_1)
drop_out_layer_7   = Dropout(0.5)(dense_layer_6)
batch_norm_layer_2 = BatchNormalization()(drop_out_layer_7)

concat_layer        = Concatenate()([drop_out_layer_5, batch_norm_layer_2])
dense_layer_7       = Dense(10, activation='relu')(concat_layer)
output  = Dense(5, activation='softmax')(dense_layer_7)
model   = Model(inputs=[input_1, input_2], outputs=output)

# compile the keras model
#opt = optimizers.Adamax(lr=0.01)
opt = SGD(lr=0.001, momentum=0.9)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['acc'])

In [ ]:
print(model.summary())

In [ ]:
plot_model(model, to_file='model_plot3.png', show_shapes=True, show_layer_names=True)

In [ ]:
# Use earlystopping
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=7, min_delta=1E-3)
rlrp = ReduceLROnPlateau(monitor='val_loss', factor=0.0001, patience=5, min_delta=1E-4)

target_type = 'multi_label'
metrics = Metrics(validation_data=([X_text_train, X_cat_train], y_cat_train, target_type))

# fit the keras model on the dataset
training_history = model.fit([X_text_train, X_cat_train], y_cat_train, epochs=100, batch_size=8, verbose=1, validation_data=([X_text_test, X_cat_test], y_cat_test), callbacks=[rlrp, metrics])

In [ ]:
# evaluate the keras model
_, train_accuracy = model.evaluate([X_text_train, X_cat_train], y_cat_train, batch_size=8, verbose=0)
_, test_accuracy = model.evaluate([X_text_test, X_cat_test], y_cat_test, batch_size=8, verbose=0)

print('Train accuracy: %.2f' % (train_accuracy*100))
print('Test accuracy: %.2f' % (test_accuracy*100))

In [ ]:
accuracy, precision, recall, f1 = get_classification_metrics(model, [X_text_test, X_cat_test], y_cat_test, target_type)
print('Accuracy: %f' % accuracy)
print('Precision: %f' % precision)
print('Recall: %f' % recall)
print('F1 score: %f' % f1)

In [ ]:
epochs = range(len(training_history.history['loss'])) # Get number of epochs

# plot loss learning curves
plt.plot(epochs, training_history.history['loss'], label = 'train')
plt.plot(epochs, training_history.history['val_loss'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation loss')

Above one is good fit, it is identified by a training and validation loss that decreases to a point of stability with a minimal gap between the two final loss values. The loss of the model will almost always be lower on the training dataset than the validation dataset.

In [ ]:
# plot accuracy learning curves
plt.plot(epochs, training_history.history['acc'], label = 'train')
plt.plot(epochs, training_history.history['val_acc'], label = 'test')
plt.legend(loc = 'upper right')
plt.title ('Training and validation accuracy')

We could see it accuracy continually rise during training. As expected, we see the learning curves for accuracy on the test dataset plateau, indicating that the model has no longer overfit the training dataset and it is generalized model.

[Table of Contents](#table-of-contents)

<a id="conclusion"></a>
## 13. Conclusion

1.	Able to predict the accident level with a test accuracy of 73.81% and f1-score of 73.81%
2.	We have seven duplicate values in this dataset and dropped those duplicate values.
3.	We have no outliers in this dataset.
4.	We have no missing values in this dataset.
5.	Extracted the day, month and year from Date column and created new features such as weekday, weekofyear and seasons.
6.	Target variable – ‘Accident Level’ distribution is not equal (I: 309, II: 40, III: 31, IV: 30, V: 8).
7.	Class imbalance issue is handled using below methods and found out that, for this particular dataset, with original data we have achieved the better results.

  a.	Resampling techniques: Oversampling minority class

  b.	SMOTE: Generate synthetic samples
8.	By comparing the results from all ML methods with original data, we can select the best method as AdaBoost classifier with f1-score 65.38% with original data.

9. **Bootstrap sampling with RandomForestClassifier model with an accuracy of 66.7% - 77.6% is our best model.**

10. Bootstrap sampling with AdaBoostClassifier model with an accuracy of 51.7% - 76.8% is our best model.

11. Bootstrap sampling with XGBClassifier model with an accuracy of 63.5% - 74.8% is our best model.

12.	Explored below options in Neural Networks.

  a.	Convert Classification to Numerical problem: achieved a test accuracy of 53.57% which is a bad result.

  b.	Multiclass classification - Target variable - One hot encoded: achieved a test accuracy of 73.81% and f1-score of 73.81% with original data + TF-IDF features from accident description column.

  c.	Create a model with Text inputs (accident description alone) only: surprisingly achieved a test accuracy of 73.81% and f1-score of 73.81% with original data.

  d.	Create a model with Categorical features only: achieved a test accuracy of 73.81% and f1-score of 72.28% with original data.

  e.	Create a model with Multiple Inputs (concatenated the layers from text input model and categorical features input model): surprisingly achieved a test accuracy of 73.81% and f1-score of 73.81% with original data.

13.	**Finally bidirectional LSTM model can be considered to productionalized the model and predict the accident level.**


[Table of Contents](#table-of-contents)

<a id="recommendations"></a>
## 14. Recommendations

*	In this project, we discovered that the main causes of accidents are mistakes in hand-operation and time-related factor.

*	To reduce the occurrences of accidents, more stringent safety standards in hand-operation will be needed in period when many accidents occur.

*	I realized that the detail information of accidents like 'Description' is so useful to analyze the cause.

*	With more detailed information such as machining data (ex. CNC, Current, Voltage) in plants, weather information, employee's personal data (ex. age, experience in the industry sector, work performance ), we can clarify the cause of accidents more correctly.

*	With more number of observations than current number of records = 425 so that we can feed more data into ML/ANN/NLP models to train, evaluate the performance of those models and get the better results.

*	There are quite a lot of Critical risk descriptions, but with the help of SME we can decide whether this column has outliers or not and also SME can help us in understanding the data better.


[Table of Contents](#table-of-contents)

<a id="limitations"></a>
## 15. Limitations

*	We have less number of observations to analyse the cause of accidents correctly and rather we should collect more number of observations to get better results.
*	Less number of features available in dataset.
*	Lack of access to quality data.

**Where does our model fall short in the real world?**

*	Once we deploy the finalised model in Production, we might get less f1-score as compared to productionalized model results.

*	Since we are predicting the accident level, we need to be 100% sure or at least close to 100% so that we can prevent the lot of accidents in industry.

**What can you do to enhance the solution?** -- Need to work on limitations.


<a id="closing-reflections"></a>
## 16. Closing Reflections

**What did we learned from the process?**

*	How to work on Data Science project to end-to-end.
*	How to handle class imbalance data set.
*	How to build different ANN model architectures for handling multi-class classification problems.
*	How to build different NLP architectures for handling both numerical and text data.

**What I do differently next time?**

*	Perhaps I will explore more feature engineering and feature selection techniques.
*	I will build the real chatbot using Streamlit or some other applications.


[Table of Contents](#table-of-contents)

<a id="references"></a>
## 17. References

*	Good EDA Notebook

  https://www.kaggle.com/koheimuramatsu/industrial-accident-causal-analysis

*	Holoviews plot tips

  http://holoviews.org/user_guide/Customizing_Plots.html

*	Scikit-Learn API

  https://scikit-learn.org/stable/

*	Keras API

  https://keras.io/api/

*	Streamlit API

  https://docs.streamlit.io/en/stable/api.html
  https://share.streamlit.io/daniellewisdl/streamlit-cheat-sheet/app.py

*	Heroku with Python

  https://devcenter.heroku.com/articles/getting-started-with-python
